# Task C — Selective Attention & Interference Resistance

**Track:** Attention — Selective Attention
**Benchmark:** CogAttention v1.0
**Subtasks:** `selective` (Signal-in-Noise), `stroop` (Semantic Stroop), `flanker` (Flanker Interference)

---

## What This Notebook Does

This notebook benchmarks an LLM's **selective attention** — its ability to focus on task-relevant information while suppressing distractors, resist automatic correction impulses (Stroop-like interference), and extract a target from semantically similar flanking content.

### Subtask Breakdown

| Subtask | Paradigm | What It Measures |
|---------|----------|-----------------|
| **Signal-in-Noise (SiN)** | Dichotic Listening / Cherry (1953) | Embeds target information among high volumes of irrelevant distractor text. Model must extract the signal while ignoring noise. Distractor density and semantic similarity scale with difficulty. |
| **Semantic Stroop** | Stroop Task (Stroop, 1935) | Presents deliberately incorrect statements and asks the model to report what was *stated* (not what is *true*). Tests whether the model can suppress its correction reflex. |
| **Flanker** | Eriksen Flanker Task (1974) | Surrounds a target passage with semantically similar but conflicting flanker passages. Model must extract info from the correct (cued) passage only. |

### Cognitive Science Grounding

- **Selective attention / cocktail party effect** (Cherry, 1953; Broadbent, 1958): Humans can attend to one voice in a noisy room; LLMs must similarly filter signal from noise in text.
- **Stroop interference** (Stroop, 1935; MacLeod, 1991): Automatic processing (reading/correction) conflicts with controlled processing (reporting stated content). A fundamental test of cognitive control.
- **Flanker compatibility** (Eriksen & Eriksen, 1974): Nearby distractors that are similar to the target create more interference than dissimilar ones — tests spatial/contextual selectivity.

### Difficulty Scaling

| Level | Signal-in-Noise | Semantic Stroop | Flanker |
|-------|----------------|-----------------|---------|
| Easy | Low noise, distinct signal | Obvious factual errors | Dissimilar flankers, clear cue |
| Medium | Moderate noise | Subtle errors | Moderately similar flankers |
| Hard | High noise, semantic overlap | Plausible-but-wrong statements | Highly similar flankers |
| Expert | Very high noise ratio | Near-true statements | Near-identical flankers, weak cue |
| Frontier | Adversarial distractors | Statements that "should" be corrected | Flankers share key phrases with target |

### Scoring

SDK assertion pass rate = per-element accuracy. Each extracted signal, reported-as-stated value, and flanker-filtered answer is a separate assertion.

---

`<!-- COGATTENTION-BENCH-CANARY-4C07B9298031 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Selective Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_selective(response, gold, kbench):
    for signal in gold["signals"]:
        pattern = rf"(?i){_escape_for_regex(signal)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Should extract signal value '{signal}'"
        )


def run_assertions_stroop(response, gold, kbench):
    for idx_str, gold_val in gold["answers"].items():
        pattern = rf"(?i){re.escape(idx_str)}\s*[.):\-]\s*.*{_escape_for_regex(gold_val)}"
        trap = gold["traps"].get(idx_str)
        trap_note = f" (trap: '{trap}')" if trap else ""
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Item {idx_str} should be '{gold_val}'{trap_note}"
        )


def run_assertions_flanker(response, gold, kbench):
    gold_val = gold["gold_value"]
    pattern = rf"(?i){_escape_for_regex(gold_val)}"
    kbench.assertions.assert_contains_regex(
        pattern, response,
        expectation=f"Should extract target value '{gold_val}'"
    )


print("CogAttention helpers loaded")
print(f"Task types: ['selective', 'stroop', 'flanker']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_selective")
def cogattention_selective(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention selective task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_selective(response, gold, kbench)


@kbench.task(name="cogattention_stroop")
def cogattention_stroop(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention stroop task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_stroop(response, gold, kbench)


@kbench.task(name="cogattention_flanker")
def cogattention_flanker(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention flanker task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_flanker(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "selective_easy_000",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] Preliminary estimates suggest costs near $8,841.47.\n[Source B (unverified)] Unconfirmed sources report gains of $9,529.02.\n[Source A (verified)] Confirmed quarterly earnings reached $9,512.96.\n[Source B (unverified)] Preliminary estimates suggest costs near $8,948.23.\n[Source A (verified)] Confirmed quarterly earnings reached $6,474.86.\n[Source A (verified)] Confirmed quarterly earnings reached $3,201.33.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3,201.33\", \"9,512.96\", \"6,474.86\"], \"distractors\": [\"9,529.02\", \"8,841.47\", \"8,948.23\"]}"
 },
 {
  "task_id": "selective_easy_001",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] Speculation puts the figure at $213.58.\n[Source A (verified)] Confirmed quarterly earnings reached $5,163.45.\n[Source B (unverified)] An unaudited draft mentions liabilities of $3,299.56.\n[Source A (verified)] The verified audit found revenue of $2,217.28.\n[Source B (unverified)] Preliminary estimates suggest costs near $2,125.71.\n[Source A (verified)] The validated balance sheet shows assets of $1,951.70.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"2,217.28\", \"5,163.45\", \"1,951.70\"], \"distractors\": [\"3,299.56\", \"213.58\", \"2,125.71\"]}"
 },
 {
  "task_id": "selective_easy_002",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source A (verified)] The accredited lab measured concentration at 148.5 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 73.7 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 68.0 mg/L.\n[Source A (verified)] Quality-controlled testing showed 114.5 mg/L.\n[Source B (unverified)] An uncertified instrument read 52.0 mg/L.\n[Source A (verified)] Certified analysis confirmed 399.7 mg/L of the compound.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"114.5\", \"399.7\", \"148.5\"], \"distractors\": [\"68.0\", \"73.7\", \"52.0\"]}"
 },
 {
  "task_id": "selective_easy_003",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] A passerby estimated the temperature at 9.2°C.\n[Source A (verified)] Official meteorological data shows 9.8°C at noon.\n[Source B (unverified)] An unverified weather app displayed 3.3°C.\n[Source B (unverified)] An unverified weather app displayed 0.4°C.\n[Source A (verified)] The verified sensor reading was -12.7°C.\n[Source A (verified)] The verified sensor reading was 2.5°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-12.7\", \"2.5\", \"9.8\"], \"distractors\": [\"0.4\", \"3.3\", \"9.2\"]}"
 },
 {
  "task_id": "selective_easy_004",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source A (verified)] Quality-controlled testing showed 473.4 mg/L.\n[Source B (unverified)] An uncertified instrument read 152.6 mg/L.\n[Source A (verified)] Quality-controlled testing showed 326.4 mg/L.\n[Source B (unverified)] Preliminary screening suggested 320.7 mg/L.\n[Source A (verified)] The peer-reviewed measurement yielded 351.1 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 21.2 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"326.4\", \"351.1\", \"473.4\"], \"distractors\": [\"21.2\", \"320.7\", \"152.6\"]}"
 },
 {
  "task_id": "selective_easy_005",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] The expired reagent test showed 233.2 mg/L.\n[Source A (verified)] The peer-reviewed measurement yielded 15.6 mg/L.\n[Source A (verified)] Quality-controlled testing showed 218.1 mg/L.\n[Source B (unverified)] A field test kit indicated approximately 457.4 mg/L.\n[Source B (unverified)] Preliminary screening suggested 99.4 mg/L.\n[Source A (verified)] The peer-reviewed measurement yielded 26.5 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"26.5\", \"218.1\", \"15.6\"], \"distractors\": [\"457.4\", \"233.2\", \"99.4\"]}"
 },
 {
  "task_id": "selective_easy_006",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] An uncalibrated thermometer showed 17.2°C.\n[Source B (unverified)] An uncalibrated thermometer showed 1.3°C.\n[Source B (unverified)] Social media posts claimed it felt like 8.0°C.\n[Source A (verified)] The calibrated station recorded a temperature of 3.3°C.\n[Source A (verified)] The verified sensor reading was 31.1°C.\n[Source A (verified)] The verified sensor reading was 33.6°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3.3\", \"31.1\", \"33.6\"], \"distractors\": [\"17.2\", \"8.0\", \"1.3\"]}"
 },
 {
  "task_id": "selective_easy_007",
  "task_type": "selective",
  "difficulty": "Easy",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements are tagged with [Source A (verified)] or [Source B (unverified)].\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\n[Source B (unverified)] A passerby estimated the temperature at 33.2°C.\n[Source B (unverified)] An uncalibrated thermometer showed 9.6°C.\n[Source A (verified)] The verified sensor reading was 1.8°C.\n[Source A (verified)] The verified sensor reading was 11.3°C.\n[Source B (unverified)] An unverified weather app displayed -10.3°C.\n[Source A (verified)] The calibrated station recorded a temperature of 31.8°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"11.3\", \"1.8\", \"31.8\"], \"distractors\": [\"-10.3\", \"33.2\", \"9.6\"]}"
 },
 {
  "task_id": "selective_medium_008",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that Preliminary estimates suggest costs near $8,247.07.\nThe unverified source claims Unconfirmed sources report gains of $3,915.05.\nAccording to Source A, Confirmed quarterly earnings reached $9,689.27.\nAccording to Source B, Unconfirmed sources report gains of $4,846.58.\nSource A reports that Confirmed quarterly earnings reached $4,643.39.\nAccording to Source B, Unconfirmed sources report gains of $8,614.07.\nSource A reports that The validated balance sheet shows assets of $7,978.79.\nSource A reports that According to the certified report, expenses totaled $4,699.28.\nThe unverified source claims Speculation puts the figure at $3,307.16.\nThe unverified source claims An unaudited draft mentions liabilities of $8,036.15.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"4,643.39\", \"7,978.79\", \"4,699.28\", \"9,689.27\"], \"distractors\": [\"3,307.16\", \"8,036.15\", \"8,247.07\", \"3,915.05\", \"4,846.58\", \"8,614.07\"]}"
 },
 {
  "task_id": "selective_medium_009",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe unverified source claims An uncertified instrument read 375.6 mg/L.\nThe verified source states The peer-reviewed measurement yielded 66.7 mg/L.\nThe verified source states The accredited lab measured concentration at 371.7 mg/L.\nThe unverified source claims A field test kit indicated approximately 31.4 mg/L.\nAccording to Source B, A field test kit indicated approximately 328.1 mg/L.\nAccording to Source A, The accredited lab measured concentration at 158.7 mg/L.\nSource A reports that Certified analysis confirmed 122.2 mg/L of the compound.\nThe unverified source claims A field test kit indicated approximately 449.8 mg/L.\nSource B suggests that An uncertified instrument read 128.2 mg/L.\nSource B suggests that A field test kit indicated approximately 416.6 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"122.2\", \"158.7\", \"66.7\", \"371.7\"], \"distractors\": [\"416.6\", \"328.1\", \"375.6\", \"128.2\", \"31.4\", \"449.8\"]}"
 },
 {
  "task_id": "selective_medium_010",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to Source B, The expired reagent test showed 136.2 mg/L.\nAccording to Source B, Preliminary screening suggested 31.2 mg/L.\nSource B suggests that The expired reagent test showed 108.8 mg/L.\nSource A reports that Quality-controlled testing showed 81.8 mg/L.\nAccording to Source A, Quality-controlled testing showed 380.0 mg/L.\nThe unverified source claims A field test kit indicated approximately 185.5 mg/L.\nThe unverified source claims The expired reagent test showed 89.5 mg/L.\nThe unverified source claims The expired reagent test showed 2.8 mg/L.\nAccording to Source A, The peer-reviewed measurement yielded 83.3 mg/L.\nThe verified source states The accredited lab measured concentration at 378.0 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"378.0\", \"380.0\", \"81.8\", \"83.3\"], \"distractors\": [\"136.2\", \"31.2\", \"185.5\", \"108.8\", \"89.5\", \"2.8\"]}"
 },
 {
  "task_id": "selective_medium_011",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that An unaudited draft mentions liabilities of $2,017.29.\nAccording to Source B, Speculation puts the figure at $2,233.80.\nThe unverified source claims Speculation puts the figure at $2,019.67.\nAccording to Source A, According to the certified report, expenses totaled $2,462.95.\nAccording to Source A, The verified audit found revenue of $9,468.04.\nSource A reports that Confirmed quarterly earnings reached $124.03.\nSource B suggests that Preliminary estimates suggest costs near $3,874.68.\nSource A reports that The validated balance sheet shows assets of $6,927.26.\nSource B suggests that Unconfirmed sources report gains of $3,925.74.\nThe unverified source claims Unconfirmed sources report gains of $9,129.98.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"2,462.95\", \"9,468.04\", \"6,927.26\", \"124.03\"], \"distractors\": [\"2,233.80\", \"2,017.29\", \"3,925.74\", \"3,874.68\", \"2,019.67\", \"9,129.98\"]}"
 },
 {
  "task_id": "selective_medium_012",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource A reports that The validated balance sheet shows assets of $2,557.13.\nAccording to Source A, The verified audit found revenue of $3,337.46.\nThe unverified source claims Preliminary estimates suggest costs near $4,760.08.\nAccording to Source B, Speculation puts the figure at $8,839.24.\nSource A reports that Confirmed quarterly earnings reached $3,215.74.\nThe verified source states The validated balance sheet shows assets of $2,462.29.\nThe unverified source claims Unconfirmed sources report gains of $9,173.31.\nSource B suggests that An unaudited draft mentions liabilities of $5,907.31.\nSource B suggests that Preliminary estimates suggest costs near $6,307.03.\nAccording to Source B, An unaudited draft mentions liabilities of $8,624.97.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3,215.74\", \"2,462.29\", \"3,337.46\", \"2,557.13\"], \"distractors\": [\"9,173.31\", \"4,760.08\", \"5,907.31\", \"8,624.97\", \"6,307.03\", \"8,839.24\"]}"
 },
 {
  "task_id": "selective_medium_013",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that A passerby estimated the temperature at 4.8°C.\nAccording to Source B, An uncalibrated thermometer showed 21.8°C.\nSource A reports that Official meteorological data shows -3.7°C at noon.\nAccording to Source B, An unverified weather app displayed 11.7°C.\nSource B suggests that A passerby estimated the temperature at 11.5°C.\nSource A reports that The calibrated station recorded a temperature of 30.3°C.\nThe unverified source claims An uncalibrated thermometer showed 34.0°C.\nSource A reports that The calibrated station recorded a temperature of 12.0°C.\nSource B suggests that Social media posts claimed it felt like 38.4°C.\nThe verified source states According to the certified gauge, pressure-adjusted temperature was 23.2°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"30.3\", \"23.2\", \"12.0\", \"-3.7\"], \"distractors\": [\"21.8\", \"11.5\", \"34.0\", \"4.8\", \"11.7\", \"38.4\"]}"
 },
 {
  "task_id": "selective_medium_014",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe unverified source claims Social media posts claimed it felt like 14.6°C.\nSource B suggests that An unverified weather app displayed -13.0°C.\nThe unverified source claims An unverified weather app displayed 23.4°C.\nAccording to Source B, Social media posts claimed it felt like 19.7°C.\nSource B suggests that Social media posts claimed it felt like 30.1°C.\nAccording to Source A, The calibrated station recorded a temperature of 37.0°C.\nSource B suggests that Social media posts claimed it felt like -9.9°C.\nSource A reports that Official meteorological data shows -2.0°C at noon.\nSource A reports that Official meteorological data shows 24.5°C at noon.\nAccording to Source A, The calibrated station recorded a temperature of 31.9°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-2.0\", \"24.5\", \"37.0\", \"31.9\"], \"distractors\": [\"-9.9\", \"14.6\", \"23.4\", \"19.7\", \"-13.0\", \"30.1\"]}"
 },
 {
  "task_id": "selective_medium_015",
  "task_type": "selective",
  "difficulty": "Medium",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nStatements mention their source inline (Source A is verified, Source B is unverified).\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSource B suggests that An uncertified instrument read 141.4 mg/L.\nThe unverified source claims Preliminary screening suggested 468.4 mg/L.\nSource B suggests that An uncertified instrument read 96.2 mg/L.\nAccording to Source A, The accredited lab measured concentration at 170.1 mg/L.\nThe unverified source claims An uncertified instrument read 54.5 mg/L.\nThe verified source states Certified analysis confirmed 104.1 mg/L of the compound.\nAccording to Source B, A field test kit indicated approximately 29.5 mg/L.\nSource A reports that Quality-controlled testing showed 402.4 mg/L.\nSource A reports that The peer-reviewed measurement yielded 333.8 mg/L.\nAccording to Source B, A field test kit indicated approximately 254.4 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"333.8\", \"170.1\", \"104.1\", \"402.4\"], \"distractors\": [\"54.5\", \"29.5\", \"141.4\", \"96.2\", \"468.4\", \"254.4\"]}"
 },
 {
  "task_id": "selective_hard_016",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nUnofficial channels report Preliminary estimates suggest costs near $5,092.67.\nPer the verified data, According to the certified report, expenses totaled $9,534.96.\nUnofficial channels report An unaudited draft mentions liabilities of $5,868.32.\nCross-referenced documentation shows The validated balance sheet shows assets of $6,132.62.\nThe audited record indicates The validated balance sheet shows assets of $7,364.67.\nUnofficial channels report Preliminary estimates suggest costs near $3,289.23.\nUnconfirmed data indicates Unconfirmed sources report gains of $9,627.12.\nUnofficial channels report Unconfirmed sources report gains of $2,725.10.\nPer the verified data, According to the certified report, expenses totaled $2,779.71.\nUnofficial channels report An unaudited draft mentions liabilities of $5,545.97.\nUnconfirmed data indicates Unconfirmed sources report gains of $873.17.\nAnecdotal evidence suggests An unaudited draft mentions liabilities of $338.75.\nThe audited record indicates The verified audit found revenue of $3,575.48.\nAnecdotal evidence suggests Speculation puts the figure at $6,444.98.\nUnconfirmed data indicates Preliminary estimates suggest costs near $3,109.99.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"3,575.48\", \"2,779.71\", \"9,534.96\", \"7,364.67\", \"6,132.62\"], \"distractors\": [\"338.75\", \"5,545.97\", \"3,109.99\", \"2,725.10\", \"873.17\", \"5,868.32\", \"5,092.67\", \"6,444.98\", \"3,289.23\", \"9,627.12\"]}"
 },
 {
  "task_id": "selective_hard_017",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnecdotal evidence suggests An unverified weather app displayed 38.6°C.\nCross-referenced documentation shows The calibrated station recorded a temperature of 7.1°C.\nUnconfirmed data indicates An unverified weather app displayed 6.7°C.\nAnecdotal evidence suggests An unverified weather app displayed -1.4°C.\nUnconfirmed data indicates An unverified weather app displayed 21.6°C.\nUnofficial channels report An uncalibrated thermometer showed 29.9°C.\nAnecdotal evidence suggests Social media posts claimed it felt like 20.0°C.\nAnecdotal evidence suggests Social media posts claimed it felt like 16.2°C.\nThe audited record indicates The verified sensor reading was 1.3°C.\nAnecdotal evidence suggests Social media posts claimed it felt like -8.8°C.\nPer the verified data, According to the certified gauge, pressure-adjusted temperature was 29.4°C.\nThe audited record indicates The calibrated station recorded a temperature of 4.3°C.\nUnconfirmed data indicates A passerby estimated the temperature at 28.8°C.\nCross-referenced documentation shows Official meteorological data shows -15.0°C at noon.\nUnofficial channels report A passerby estimated the temperature at 33.4°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"29.4\", \"-15.0\", \"4.3\", \"1.3\", \"7.1\"], \"distractors\": [\"33.4\", \"-8.8\", \"-1.4\", \"6.7\", \"16.2\", \"21.6\", \"38.6\", \"29.9\", \"28.8\", \"20.0\"]}"
 },
 {
  "task_id": "selective_hard_018",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nCross-referenced documentation shows The peer-reviewed measurement yielded 333.1 mg/L.\nThe audited record indicates The accredited lab measured concentration at 42.1 mg/L.\nUnofficial channels report The expired reagent test showed 136.7 mg/L.\nUnofficial channels report A field test kit indicated approximately 235.1 mg/L.\nThe audited record indicates Quality-controlled testing showed 440.6 mg/L.\nAnecdotal evidence suggests Preliminary screening suggested 351.0 mg/L.\nAnecdotal evidence suggests An uncertified instrument read 359.1 mg/L.\nUnconfirmed data indicates A field test kit indicated approximately 242.8 mg/L.\nUnofficial channels report Preliminary screening suggested 178.4 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 353.3 mg/L.\nUnofficial channels report The expired reagent test showed 134.1 mg/L.\nAnecdotal evidence suggests The expired reagent test showed 312.8 mg/L.\nUnconfirmed data indicates An uncertified instrument read 24.6 mg/L.\nThe audited record indicates The accredited lab measured concentration at 49.7 mg/L.\nPer the verified data, The accredited lab measured concentration at 5.6 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"440.6\", \"333.1\", \"49.7\", \"42.1\", \"5.6\"], \"distractors\": [\"134.1\", \"312.8\", \"351.0\", \"359.1\", \"353.3\", \"136.7\", \"242.8\", \"178.4\", \"24.6\", \"235.1\"]}"
 },
 {
  "task_id": "selective_hard_019",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe audited record indicates The peer-reviewed measurement yielded 239.2 mg/L.\nPer the verified data, The accredited lab measured concentration at 355.9 mg/L.\nCross-referenced documentation shows Quality-controlled testing showed 53.3 mg/L.\nUnconfirmed data indicates A field test kit indicated approximately 61.7 mg/L.\nThe audited record indicates The accredited lab measured concentration at 239.5 mg/L.\nCross-referenced documentation shows Quality-controlled testing showed 241.9 mg/L.\nUnofficial channels report Preliminary screening suggested 125.4 mg/L.\nUnofficial channels report The expired reagent test showed 64.1 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 332.7 mg/L.\nUnofficial channels report An uncertified instrument read 288.9 mg/L.\nUnofficial channels report An uncertified instrument read 353.8 mg/L.\nUnofficial channels report A field test kit indicated approximately 411.7 mg/L.\nAnecdotal evidence suggests The expired reagent test showed 473.2 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 174.4 mg/L.\nUnofficial channels report The expired reagent test showed 127.5 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"355.9\", \"53.3\", \"239.2\", \"239.5\", \"241.9\"], \"distractors\": [\"411.7\", \"125.4\", \"353.8\", \"332.7\", \"174.4\", \"64.1\", \"288.9\", \"127.5\", \"473.2\", \"61.7\"]}"
 },
 {
  "task_id": "selective_hard_020",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nUnofficial channels report Unconfirmed sources report gains of $4,369.23.\nUnconfirmed data indicates Speculation puts the figure at $7,617.62.\nAnecdotal evidence suggests Unconfirmed sources report gains of $324.22.\nUnconfirmed data indicates Unconfirmed sources report gains of $9,988.19.\nAnecdotal evidence suggests Preliminary estimates suggest costs near $5,954.72.\nUnofficial channels report Preliminary estimates suggest costs near $2,713.77.\nUnofficial channels report Speculation puts the figure at $7,532.54.\nThe audited record indicates The verified audit found revenue of $4,819.02.\nPer the verified data, The verified audit found revenue of $7,913.23.\nUnconfirmed data indicates An unaudited draft mentions liabilities of $7,942.39.\nUnconfirmed data indicates Preliminary estimates suggest costs near $6,397.21.\nCross-referenced documentation shows The validated balance sheet shows assets of $4,543.29.\nPer the verified data, The validated balance sheet shows assets of $974.85.\nPer the verified data, Confirmed quarterly earnings reached $3,303.87.\nAnecdotal evidence suggests Unconfirmed sources report gains of $9,016.10.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"4,819.02\", \"4,543.29\", \"3,303.87\", \"7,913.23\", \"974.85\"], \"distractors\": [\"324.22\", \"7,942.39\", \"9,988.19\", \"4,369.23\", \"6,397.21\", \"7,532.54\", \"2,713.77\", \"7,617.62\", \"9,016.10\", \"5,954.72\"]}"
 },
 {
  "task_id": "selective_hard_021",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nUnconfirmed data indicates An unaudited draft mentions liabilities of $6,708.89.\nUnconfirmed data indicates Unconfirmed sources report gains of $1,318.56.\nUnconfirmed data indicates Speculation puts the figure at $7,151.85.\nUnofficial channels report Unconfirmed sources report gains of $679.02.\nAnecdotal evidence suggests Unconfirmed sources report gains of $9,018.44.\nUnconfirmed data indicates An unaudited draft mentions liabilities of $5,156.55.\nCross-referenced documentation shows The validated balance sheet shows assets of $2,636.72.\nUnofficial channels report Unconfirmed sources report gains of $7,179.46.\nUnofficial channels report An unaudited draft mentions liabilities of $1,415.11.\nPer the verified data, Confirmed quarterly earnings reached $826.26.\nCross-referenced documentation shows The verified audit found revenue of $4,986.81.\nCross-referenced documentation shows According to the certified report, expenses totaled $6,583.25.\nUnconfirmed data indicates An unaudited draft mentions liabilities of $5,835.33.\nAnecdotal evidence suggests Preliminary estimates suggest costs near $635.12.\nCross-referenced documentation shows According to the certified report, expenses totaled $7,061.29.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"6,583.25\", \"2,636.72\", \"826.26\", \"4,986.81\", \"7,061.29\"], \"distractors\": [\"9,018.44\", \"5,156.55\", \"635.12\", \"679.02\", \"1,415.11\", \"1,318.56\", \"7,151.85\", \"6,708.89\", \"5,835.33\", \"7,179.46\"]}"
 },
 {
  "task_id": "selective_hard_022",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnecdotal evidence suggests Preliminary screening suggested 177.2 mg/L.\nUnconfirmed data indicates Preliminary screening suggested 416.3 mg/L.\nPer the verified data, Certified analysis confirmed 75.0 mg/L of the compound.\nUnofficial channels report An uncertified instrument read 264.0 mg/L.\nUnofficial channels report Preliminary screening suggested 194.8 mg/L.\nUnofficial channels report The expired reagent test showed 27.7 mg/L.\nCross-referenced documentation shows The peer-reviewed measurement yielded 103.1 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 194.5 mg/L.\nCross-referenced documentation shows Certified analysis confirmed 473.1 mg/L of the compound.\nUnconfirmed data indicates Preliminary screening suggested 486.0 mg/L.\nPer the verified data, The peer-reviewed measurement yielded 75.8 mg/L.\nUnconfirmed data indicates Preliminary screening suggested 406.7 mg/L.\nCross-referenced documentation shows Certified analysis confirmed 221.6 mg/L of the compound.\nAnecdotal evidence suggests The expired reagent test showed 228.2 mg/L.\nUnconfirmed data indicates An uncertified instrument read 412.3 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"75.0\", \"75.8\", \"221.6\", \"103.1\", \"473.1\"], \"distractors\": [\"194.5\", \"177.2\", \"194.8\", \"406.7\", \"264.0\", \"486.0\", \"228.2\", \"412.3\", \"416.3\", \"27.7\"]}"
 },
 {
  "task_id": "selective_hard_023",
  "task_type": "selective",
  "difficulty": "Hard",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nVerified data comes from audited/cross-referenced sources. Unverified data comes from unofficial/anecdotal sources.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nPer the verified data, The accredited lab measured concentration at 387.6 mg/L.\nUnconfirmed data indicates The expired reagent test showed 93.6 mg/L.\nUnofficial channels report An uncertified instrument read 94.0 mg/L.\nPer the verified data, Certified analysis confirmed 106.6 mg/L of the compound.\nUnofficial channels report Preliminary screening suggested 81.5 mg/L.\nAnecdotal evidence suggests An uncertified instrument read 198.2 mg/L.\nAnecdotal evidence suggests An uncertified instrument read 195.0 mg/L.\nUnconfirmed data indicates The expired reagent test showed 446.1 mg/L.\nPer the verified data, The accredited lab measured concentration at 350.2 mg/L.\nUnconfirmed data indicates A field test kit indicated approximately 221.7 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 451.8 mg/L.\nAnecdotal evidence suggests A field test kit indicated approximately 449.0 mg/L.\nPer the verified data, The peer-reviewed measurement yielded 429.8 mg/L.\nUnconfirmed data indicates An uncertified instrument read 110.5 mg/L.\nThe audited record indicates The peer-reviewed measurement yielded 75.0 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"429.8\", \"106.6\", \"387.6\", \"350.2\", \"75.0\"], \"distractors\": [\"446.1\", \"221.7\", \"451.8\", \"195.0\", \"449.0\", \"93.6\", \"110.5\", \"198.2\", \"81.5\", \"94.0\"]}"
 },
 {
  "task_id": "selective_expert_024",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nIt has been suggested that An unverified weather app displayed 12.5°C.\nRecords show The calibrated station recorded a temperature of 5.3°C.\nReportedly, An unverified weather app displayed -11.6°C.\nSome estimates place A passerby estimated the temperature at -3.7°C.\nDocumentation confirms According to the certified gauge, pressure-adjusted temperature was -4.7°C.\nSome estimates place An unverified weather app displayed -10.3°C.\nThe data indicates The verified sensor reading was 21.6°C.\nSome estimates place An uncalibrated thermometer showed -4.3°C.\nThe data indicates The verified sensor reading was 16.1°C.\nIt has been suggested that An unverified weather app displayed 20.6°C.\nReportedly, A passerby estimated the temperature at 26.0°C.\nSome estimates place A passerby estimated the temperature at -0.6°C.\nSome estimates place An unverified weather app displayed 22.3°C.\nIt has been suggested that A passerby estimated the temperature at -11.7°C.\nReportedly, An uncalibrated thermometer showed 34.9°C.\nDocumentation confirms Official meteorological data shows -14.9°C at noon.\nThe data indicates The calibrated station recorded a temperature of 13.9°C.\nSome estimates place A passerby estimated the temperature at -8.6°C.\nReportedly, Social media posts claimed it felt like 39.4°C.\nIt has been suggested that A passerby estimated the temperature at 33.5°C.\nReportedly, An uncalibrated thermometer showed 39.2°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"21.6\", \"13.9\", \"16.1\", \"-14.9\", \"5.3\", \"-4.7\"], \"distractors\": [\"22.3\", \"34.9\", \"33.5\", \"-0.6\", \"12.5\", \"-8.6\", \"20.6\", \"-11.6\", \"-10.3\", \"39.2\", \"-3.7\", \"-4.3\", \"39.4\", \"26.0\", \"-11.7\"]}"
 },
 {
  "task_id": "selective_expert_025",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nReportedly, Unconfirmed sources report gains of $5,350.77.\nSome estimates place Preliminary estimates suggest costs near $5,839.22.\nIt has been suggested that Unconfirmed sources report gains of $5,682.30.\nRecords show According to the certified report, expenses totaled $7,291.17.\nSome estimates place Preliminary estimates suggest costs near $144.68.\nReportedly, An unaudited draft mentions liabilities of $970.36.\nSome estimates place Unconfirmed sources report gains of $1,725.22.\nReportedly, Speculation puts the figure at $6,899.39.\nReportedly, Preliminary estimates suggest costs near $2,243.98.\nThe data indicates The validated balance sheet shows assets of $8,972.86.\nSome estimates place Unconfirmed sources report gains of $6,398.08.\nReportedly, Unconfirmed sources report gains of $1,529.14.\nRecords show According to the certified report, expenses totaled $5,411.24.\nRecords show The verified audit found revenue of $2,423.93.\nReportedly, An unaudited draft mentions liabilities of $1,825.54.\nRecords show The verified audit found revenue of $8,031.12.\nReportedly, An unaudited draft mentions liabilities of $3,081.84.\nSome estimates place An unaudited draft mentions liabilities of $3,155.84.\nRecords show The validated balance sheet shows assets of $2,113.92.\nReportedly, An unaudited draft mentions liabilities of $6,171.90.\nIt has been suggested that Unconfirmed sources report gains of $4,966.00.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"2,113.92\", \"7,291.17\", \"5,411.24\", \"8,031.12\", \"8,972.86\", \"2,423.93\"], \"distractors\": [\"3,081.84\", \"6,171.90\", \"1,725.22\", \"2,243.98\", \"5,839.22\", \"4,966.00\", \"5,350.77\", \"5,682.30\", \"1,825.54\", \"970.36\", \"3,155.84\", \"1,529.14\", \"6,398.08\", \"6,899.39\", \"144.68\"]}"
 },
 {
  "task_id": "selective_expert_026",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nSome estimates place Preliminary screening suggested 235.7 mg/L.\nReportedly, Preliminary screening suggested 116.8 mg/L.\nIt has been suggested that Preliminary screening suggested 62.1 mg/L.\nIt has been suggested that An uncertified instrument read 102.4 mg/L.\nThe data indicates Quality-controlled testing showed 327.3 mg/L.\nIt has been suggested that A field test kit indicated approximately 240.6 mg/L.\nReportedly, A field test kit indicated approximately 397.0 mg/L.\nReportedly, An uncertified instrument read 362.3 mg/L.\nSome estimates place An uncertified instrument read 311.9 mg/L.\nIt has been suggested that A field test kit indicated approximately 475.9 mg/L.\nThe data indicates Certified analysis confirmed 237.6 mg/L of the compound.\nReportedly, A field test kit indicated approximately 435.5 mg/L.\nReportedly, The expired reagent test showed 459.3 mg/L.\nIt has been suggested that Preliminary screening suggested 319.0 mg/L.\nIt has been suggested that An uncertified instrument read 129.6 mg/L.\nDocumentation confirms The accredited lab measured concentration at 73.0 mg/L.\nDocumentation confirms Certified analysis confirmed 470.6 mg/L of the compound.\nIt has been suggested that A field test kit indicated approximately 94.2 mg/L.\nDocumentation confirms Certified analysis confirmed 210.8 mg/L of the compound.\nReportedly, A field test kit indicated approximately 4.7 mg/L.\nDocumentation confirms Quality-controlled testing showed 407.9 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"73.0\", \"327.3\", \"470.6\", \"407.9\", \"210.8\", \"237.6\"], \"distractors\": [\"62.1\", \"319.0\", \"235.7\", \"475.9\", \"129.6\", \"4.7\", \"435.5\", \"362.3\", \"240.6\", \"459.3\", \"397.0\", \"116.8\", \"94.2\", \"311.9\", \"102.4\"]}"
 },
 {
  "task_id": "selective_expert_027",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nRecords show According to the certified gauge, pressure-adjusted temperature was 29.8°C.\nSome estimates place A passerby estimated the temperature at 21.5°C.\nRecords show According to the certified gauge, pressure-adjusted temperature was -6.5°C.\nDocumentation confirms The verified sensor reading was 23.6°C.\nSome estimates place An unverified weather app displayed 2.4°C.\nIt has been suggested that An uncalibrated thermometer showed 34.0°C.\nReportedly, An unverified weather app displayed -8.9°C.\nIt has been suggested that An uncalibrated thermometer showed 1.7°C.\nIt has been suggested that A passerby estimated the temperature at 35.2°C.\nIt has been suggested that A passerby estimated the temperature at 33.9°C.\nThe data indicates Official meteorological data shows -6.1°C at noon.\nReportedly, An uncalibrated thermometer showed 12.6°C.\nSome estimates place Social media posts claimed it felt like 24.7°C.\nThe data indicates According to the certified gauge, pressure-adjusted temperature was -8.1°C.\nIt has been suggested that An unverified weather app displayed 36.6°C.\nIt has been suggested that An unverified weather app displayed 27.4°C.\nReportedly, A passerby estimated the temperature at 18.0°C.\nSome estimates place An unverified weather app displayed 19.6°C.\nSome estimates place An unverified weather app displayed 33.8°C.\nSome estimates place Social media posts claimed it felt like 38.1°C.\nThe data indicates Official meteorological data shows 18.7°C at noon.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"29.8\", \"18.7\", \"-8.1\", \"-6.5\", \"-6.1\", \"23.6\"], \"distractors\": [\"38.1\", \"2.4\", \"19.6\", \"27.4\", \"21.5\", \"35.2\", \"12.6\", \"36.6\", \"-8.9\", \"18.0\", \"34.0\", \"33.9\", \"1.7\", \"33.8\", \"24.7\"]}"
 },
 {
  "task_id": "selective_expert_028",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nIt has been suggested that An uncertified instrument read 126.3 mg/L.\nSome estimates place The expired reagent test showed 215.4 mg/L.\nRecords show The peer-reviewed measurement yielded 21.2 mg/L.\nSome estimates place A field test kit indicated approximately 237.6 mg/L.\nDocumentation confirms The peer-reviewed measurement yielded 231.8 mg/L.\nReportedly, Preliminary screening suggested 137.6 mg/L.\nReportedly, A field test kit indicated approximately 114.3 mg/L.\nThe data indicates Quality-controlled testing showed 126.9 mg/L.\nIt has been suggested that An uncertified instrument read 398.0 mg/L.\nRecords show Quality-controlled testing showed 345.4 mg/L.\nIt has been suggested that A field test kit indicated approximately 434.2 mg/L.\nThe data indicates The peer-reviewed measurement yielded 452.0 mg/L.\nReportedly, A field test kit indicated approximately 13.3 mg/L.\nReportedly, An uncertified instrument read 459.0 mg/L.\nReportedly, A field test kit indicated approximately 320.9 mg/L.\nReportedly, Preliminary screening suggested 385.8 mg/L.\nDocumentation confirms Certified analysis confirmed 355.2 mg/L of the compound.\nSome estimates place A field test kit indicated approximately 332.2 mg/L.\nIt has been suggested that Preliminary screening suggested 49.9 mg/L.\nReportedly, An uncertified instrument read 412.2 mg/L.\nIt has been suggested that An uncertified instrument read 63.7 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"231.8\", \"126.9\", \"355.2\", \"345.4\", \"452.0\", \"21.2\"], \"distractors\": [\"459.0\", \"412.2\", \"114.3\", \"13.3\", \"63.7\", \"385.8\", \"434.2\", \"237.6\", \"137.6\", \"332.2\", \"398.0\", \"49.9\", \"320.9\", \"126.3\", \"215.4\"]}"
 },
 {
  "task_id": "selective_expert_029",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nReportedly, An uncalibrated thermometer showed 21.6°C.\nRecords show According to the certified gauge, pressure-adjusted temperature was -2.4°C.\nIt has been suggested that An unverified weather app displayed -14.9°C.\nIt has been suggested that An unverified weather app displayed -4.6°C.\nReportedly, An uncalibrated thermometer showed 29.6°C.\nReportedly, Social media posts claimed it felt like -0.1°C.\nRecords show According to the certified gauge, pressure-adjusted temperature was 38.0°C.\nIt has been suggested that An unverified weather app displayed 22.9°C.\nReportedly, Social media posts claimed it felt like -2.6°C.\nIt has been suggested that An unverified weather app displayed -10.2°C.\nIt has been suggested that A passerby estimated the temperature at 21.0°C.\nRecords show Official meteorological data shows 6.0°C at noon.\nSome estimates place An unverified weather app displayed -7.3°C.\nThe data indicates The verified sensor reading was 41.3°C.\nIt has been suggested that An unverified weather app displayed 31.4°C.\nDocumentation confirms The calibrated station recorded a temperature of 30.0°C.\nThe data indicates The verified sensor reading was 35.2°C.\nIt has been suggested that A passerby estimated the temperature at 12.1°C.\nSome estimates place A passerby estimated the temperature at 26.8°C.\nReportedly, A passerby estimated the temperature at 34.2°C.\nReportedly, An unverified weather app displayed 33.6°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"38.0\", \"30.0\", \"35.2\", \"41.3\", \"6.0\", \"-2.4\"], \"distractors\": [\"33.6\", \"12.1\", \"26.8\", \"22.9\", \"-10.2\", \"29.6\", \"-4.6\", \"-0.1\", \"-2.6\", \"-14.9\", \"-7.3\", \"21.6\", \"34.2\", \"31.4\", \"21.0\"]}"
 },
 {
  "task_id": "selective_expert_030",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe data indicates Certified analysis confirmed 394.1 mg/L of the compound.\nReportedly, Preliminary screening suggested 162.8 mg/L.\nRecords show Quality-controlled testing showed 457.7 mg/L.\nReportedly, The expired reagent test showed 358.5 mg/L.\nIt has been suggested that A field test kit indicated approximately 265.5 mg/L.\nSome estimates place The expired reagent test showed 446.2 mg/L.\nRecords show The peer-reviewed measurement yielded 489.2 mg/L.\nThe data indicates The peer-reviewed measurement yielded 482.9 mg/L.\nIt has been suggested that A field test kit indicated approximately 105.9 mg/L.\nDocumentation confirms The accredited lab measured concentration at 380.5 mg/L.\nReportedly, The expired reagent test showed 390.4 mg/L.\nReportedly, Preliminary screening suggested 352.3 mg/L.\nIt has been suggested that Preliminary screening suggested 217.5 mg/L.\nSome estimates place A field test kit indicated approximately 439.1 mg/L.\nIt has been suggested that An uncertified instrument read 168.9 mg/L.\nReportedly, An uncertified instrument read 464.2 mg/L.\nIt has been suggested that A field test kit indicated approximately 498.9 mg/L.\nDocumentation confirms Quality-controlled testing showed 7.6 mg/L.\nReportedly, Preliminary screening suggested 165.3 mg/L.\nSome estimates place An uncertified instrument read 327.5 mg/L.\nReportedly, Preliminary screening suggested 115.0 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"489.2\", \"482.9\", \"394.1\", \"457.7\", \"380.5\", \"7.6\"], \"distractors\": [\"358.5\", \"446.2\", \"165.3\", \"327.5\", \"352.3\", \"265.5\", \"498.9\", \"168.9\", \"439.1\", \"105.9\", \"464.2\", \"390.4\", \"162.8\", \"115.0\", \"217.5\"]}"
 },
 {
  "task_id": "selective_expert_031",
  "task_type": "selective",
  "difficulty": "Expert",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nYou must determine which statements come from reliable documentation vs. unconfirmed reports.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nIt has been suggested that An uncalibrated thermometer showed 0.5°C.\nThe data indicates The calibrated station recorded a temperature of 11.9°C.\nDocumentation confirms The calibrated station recorded a temperature of 20.4°C.\nIt has been suggested that Social media posts claimed it felt like 24.0°C.\nThe data indicates Official meteorological data shows 41.1°C at noon.\nReportedly, A passerby estimated the temperature at 18.5°C.\nReportedly, A passerby estimated the temperature at 35.5°C.\nReportedly, An uncalibrated thermometer showed 8.4°C.\nIt has been suggested that A passerby estimated the temperature at 17.3°C.\nSome estimates place A passerby estimated the temperature at 35.1°C.\nDocumentation confirms The calibrated station recorded a temperature of -4.1°C.\nIt has been suggested that A passerby estimated the temperature at -10.2°C.\nDocumentation confirms According to the certified gauge, pressure-adjusted temperature was 3.2°C.\nReportedly, An uncalibrated thermometer showed 23.6°C.\nThe data indicates Official meteorological data shows 10.6°C at noon.\nSome estimates place An uncalibrated thermometer showed 29.5°C.\nReportedly, An unverified weather app displayed 6.3°C.\nSome estimates place A passerby estimated the temperature at 3.7°C.\nReportedly, A passerby estimated the temperature at 7.9°C.\nIt has been suggested that Social media posts claimed it felt like 24.9°C.\nSome estimates place An uncalibrated thermometer showed 15.2°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-4.1\", \"41.1\", \"11.9\", \"20.4\", \"3.2\", \"10.6\"], \"distractors\": [\"35.5\", \"3.7\", \"7.9\", \"23.6\", \"8.4\", \"18.5\", \"29.5\", \"0.5\", \"15.2\", \"24.9\", \"6.3\", \"24.0\", \"-10.2\", \"17.3\", \"35.1\"]}"
 },
 {
  "task_id": "selective_frontier_032",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to the data, Preliminary estimates suggest costs near $4,338.90.\nAnalysis shows Preliminary estimates suggest costs near $5,439.02.\nAnalysis shows Unconfirmed sources report gains of $750.61.\nThe report states Speculation puts the figure at $3,864.53.\nThe report states According to the certified report, expenses totaled $118.65.\nAnalysis shows An unaudited draft mentions liabilities of $5,360.32.\nAnalysis shows Preliminary estimates suggest costs near $3,069.27.\nThe report states Speculation puts the figure at $977.89.\nAccording to the data, The validated balance sheet shows assets of $3,893.00.\nThe findings indicate An unaudited draft mentions liabilities of $3,122.88.\nThe report states Speculation puts the figure at $7,406.96.\nAccording to the data, Unconfirmed sources report gains of $2,219.89.\nAccording to the data, Confirmed quarterly earnings reached $8,040.32.\nThe findings indicate Speculation puts the figure at $2,441.75.\nAnalysis shows Preliminary estimates suggest costs near $149.99.\nAccording to the data, Unconfirmed sources report gains of $8,770.84.\nThe findings indicate Unconfirmed sources report gains of $4,092.61.\nAccording to the data, The validated balance sheet shows assets of $4,071.84.\nAccording to the data, Preliminary estimates suggest costs near $7,107.50.\nAnalysis shows Speculation puts the figure at $2,146.11.\nThe findings indicate The verified audit found revenue of $9,214.08.\nAnalysis shows The verified audit found revenue of $2,851.75.\nAccording to the data, According to the certified report, expenses totaled $8,607.74.\nThe findings indicate Unconfirmed sources report gains of $5,467.27.\nAccording to the data, Speculation puts the figure at $5,613.04.\nAccording to the data, An unaudited draft mentions liabilities of $2,240.47.\nThe report states Preliminary estimates suggest costs near $421.06.\nThe report states Unconfirmed sources report gains of $5,596.15.\nAnalysis shows Preliminary estimates suggest costs near $8,232.45.\nThe report states Speculation puts the figure at $3,001.69.\nAnalysis shows Unconfirmed sources report gains of $3,634.59.\nAccording to the data, An unaudited draft mentions liabilities of $5,390.65.\nThe report states According to the certified report, expenses totaled $4,418.28.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"8,040.32\", \"4,071.84\", \"2,851.75\", \"9,214.08\", \"3,893.00\", \"8,607.74\", \"118.65\", \"4,418.28\"], \"distractors\": [\"149.99\", \"3,122.88\", \"5,467.27\", \"977.89\", \"5,613.04\", \"5,390.65\", \"2,146.11\", \"3,864.53\", \"5,439.02\", \"2,219.89\", \"4,338.90\", \"7,406.96\", \"2,240.47\", \"3,634.59\", \"8,232.45\", \"4,092.61\", \"5,360.32\", \"2,441.75\", \"3,001.69\", \"7,107.50\", \"3,069.27\", \"5,596.15\", \"421.06\", \"750.61\", \"8,770.84\"]}"
 },
 {
  "task_id": "selective_frontier_033",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnalysis shows An unverified weather app displayed 8.4°C.\nAccording to the data, Social media posts claimed it felt like -8.5°C.\nAnalysis shows Social media posts claimed it felt like -4.5°C.\nThe findings indicate An unverified weather app displayed 17.6°C.\nThe report states Official meteorological data shows 24.1°C at noon.\nAnalysis shows Official meteorological data shows 13.5°C at noon.\nThe findings indicate A passerby estimated the temperature at 28.3°C.\nAnalysis shows The verified sensor reading was 8.2°C.\nThe report states A passerby estimated the temperature at -6.6°C.\nAccording to the data, A passerby estimated the temperature at 16.2°C.\nAnalysis shows A passerby estimated the temperature at 31.3°C.\nThe findings indicate The calibrated station recorded a temperature of 25.6°C.\nAccording to the data, An uncalibrated thermometer showed 40.4°C.\nThe findings indicate An unverified weather app displayed 27.7°C.\nAccording to the data, An uncalibrated thermometer showed -14.7°C.\nThe report states A passerby estimated the temperature at -4.9°C.\nAccording to the data, Social media posts claimed it felt like 1.4°C.\nThe findings indicate A passerby estimated the temperature at 5.0°C.\nAnalysis shows An unverified weather app displayed 32.1°C.\nThe findings indicate A passerby estimated the temperature at 36.3°C.\nAccording to the data, An unverified weather app displayed 5.4°C.\nAccording to the data, The calibrated station recorded a temperature of 21.2°C.\nThe report states Social media posts claimed it felt like 25.0°C.\nThe findings indicate A passerby estimated the temperature at 7.7°C.\nThe findings indicate Official meteorological data shows -9.9°C at noon.\nThe findings indicate According to the certified gauge, pressure-adjusted temperature was -4.4°C.\nThe report states Social media posts claimed it felt like 27.4°C.\nThe report states A passerby estimated the temperature at 3.0°C.\nAnalysis shows An unverified weather app displayed 11.1°C.\nThe findings indicate An unverified weather app displayed 2.5°C.\nAccording to the data, An uncalibrated thermometer showed 41.0°C.\nThe findings indicate An uncalibrated thermometer showed 1.9°C.\nThe report states The verified sensor reading was 26.9°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"13.5\", \"21.2\", \"8.2\", \"24.1\", \"-9.9\", \"26.9\", \"-4.4\", \"25.6\"], \"distractors\": [\"8.4\", \"11.1\", \"27.7\", \"16.2\", \"3.0\", \"31.3\", \"2.5\", \"36.3\", \"40.4\", \"25.0\", \"5.0\", \"27.4\", \"-8.5\", \"-4.9\", \"17.6\", \"-6.6\", \"-14.7\", \"1.9\", \"5.4\", \"1.4\", \"41.0\", \"-4.5\", \"7.7\", \"32.1\", \"28.3\"]}"
 },
 {
  "task_id": "selective_frontier_034",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nThe report states A passerby estimated the temperature at 8.2°C.\nAnalysis shows A passerby estimated the temperature at -1.6°C.\nThe findings indicate A passerby estimated the temperature at 30.9°C.\nAccording to the data, An uncalibrated thermometer showed 27.1°C.\nThe findings indicate An unverified weather app displayed 33.4°C.\nAccording to the data, An uncalibrated thermometer showed 15.4°C.\nThe report states Social media posts claimed it felt like 17.5°C.\nAnalysis shows Social media posts claimed it felt like 38.2°C.\nAccording to the data, An uncalibrated thermometer showed -10.6°C.\nAnalysis shows A passerby estimated the temperature at 27.0°C.\nAnalysis shows A passerby estimated the temperature at 21.8°C.\nAccording to the data, Social media posts claimed it felt like 40.1°C.\nAnalysis shows An uncalibrated thermometer showed -5.6°C.\nThe findings indicate According to the certified gauge, pressure-adjusted temperature was -9.2°C.\nThe findings indicate An unverified weather app displayed 25.4°C.\nAnalysis shows The verified sensor reading was 34.6°C.\nThe report states An uncalibrated thermometer showed 33.5°C.\nAccording to the data, According to the certified gauge, pressure-adjusted temperature was 28.3°C.\nThe findings indicate The verified sensor reading was 22.7°C.\nThe findings indicate An unverified weather app displayed 8.9°C.\nThe findings indicate An uncalibrated thermometer showed 17.4°C.\nAccording to the data, The calibrated station recorded a temperature of 20.5°C.\nThe report states A passerby estimated the temperature at 31.8°C.\nAnalysis shows A passerby estimated the temperature at -12.5°C.\nAnalysis shows A passerby estimated the temperature at 35.5°C.\nAnalysis shows An unverified weather app displayed 35.0°C.\nThe report states The calibrated station recorded a temperature of 39.0°C.\nAccording to the data, Social media posts claimed it felt like 38.3°C.\nAnalysis shows An uncalibrated thermometer showed 32.0°C.\nThe report states The calibrated station recorded a temperature of -0.1°C.\nAccording to the data, An uncalibrated thermometer showed -2.4°C.\nAnalysis shows Social media posts claimed it felt like -12.7°C.\nThe report states The calibrated station recorded a temperature of 19.6°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-9.2\", \"34.6\", \"22.7\", \"19.6\", \"28.3\", \"20.5\", \"39.0\", \"-0.1\"], \"distractors\": [\"31.8\", \"27.0\", \"-1.6\", \"33.4\", \"32.0\", \"25.4\", \"30.9\", \"40.1\", \"38.2\", \"33.5\", \"15.4\", \"21.8\", \"-5.6\", \"35.5\", \"17.5\", \"-2.4\", \"35.0\", \"38.3\", \"-10.6\", \"8.2\", \"17.4\", \"-12.7\", \"-12.5\", \"8.9\", \"27.1\"]}"
 },
 {
  "task_id": "selective_frontier_035",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnalysis shows Preliminary estimates suggest costs near $4,224.73.\nThe findings indicate Preliminary estimates suggest costs near $7,042.11.\nThe findings indicate Preliminary estimates suggest costs near $9,253.59.\nAccording to the data, An unaudited draft mentions liabilities of $3,087.52.\nThe report states Unconfirmed sources report gains of $9,546.47.\nThe findings indicate Preliminary estimates suggest costs near $9,224.84.\nAccording to the data, Speculation puts the figure at $8,424.27.\nThe findings indicate An unaudited draft mentions liabilities of $5,662.37.\nAnalysis shows According to the certified report, expenses totaled $8,565.59.\nThe findings indicate An unaudited draft mentions liabilities of $9,315.77.\nAnalysis shows Unconfirmed sources report gains of $5,499.74.\nAnalysis shows Confirmed quarterly earnings reached $6,535.62.\nThe findings indicate Unconfirmed sources report gains of $5,547.14.\nThe findings indicate The verified audit found revenue of $8,027.17.\nAnalysis shows An unaudited draft mentions liabilities of $1,384.81.\nAccording to the data, Preliminary estimates suggest costs near $6,452.40.\nThe report states Unconfirmed sources report gains of $7,472.70.\nAnalysis shows Preliminary estimates suggest costs near $9,936.48.\nAccording to the data, Unconfirmed sources report gains of $4,036.71.\nThe findings indicate Preliminary estimates suggest costs near $808.40.\nAccording to the data, Preliminary estimates suggest costs near $7,072.66.\nAnalysis shows According to the certified report, expenses totaled $901.04.\nAccording to the data, The verified audit found revenue of $3,950.02.\nAnalysis shows Unconfirmed sources report gains of $2,874.83.\nThe report states The validated balance sheet shows assets of $3,211.04.\nAnalysis shows Unconfirmed sources report gains of $1,809.97.\nThe findings indicate Preliminary estimates suggest costs near $1,821.11.\nThe findings indicate Preliminary estimates suggest costs near $8,501.35.\nThe findings indicate Unconfirmed sources report gains of $830.01.\nAccording to the data, Speculation puts the figure at $1,398.79.\nThe report states Unconfirmed sources report gains of $647.52.\nThe findings indicate The verified audit found revenue of $4,405.88.\nThe report states According to the certified report, expenses totaled $4,578.24.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"8,565.59\", \"3,950.02\", \"3,211.04\", \"4,578.24\", \"901.04\", \"4,405.88\", \"6,535.62\", \"8,027.17\"], \"distractors\": [\"1,384.81\", \"7,072.66\", \"647.52\", \"9,546.47\", \"4,224.73\", \"9,936.48\", \"1,398.79\", \"8,501.35\", \"830.01\", \"9,253.59\", \"5,662.37\", \"4,036.71\", \"3,087.52\", \"9,224.84\", \"1,821.11\", \"5,499.74\", \"6,452.40\", \"808.40\", \"7,042.11\", \"8,424.27\", \"1,809.97\", \"5,547.14\", \"9,315.77\", \"2,874.83\", \"7,472.70\"]}"
 },
 {
  "task_id": "selective_frontier_036",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to the data, A field test kit indicated approximately 241.1 mg/L.\nThe findings indicate The expired reagent test showed 222.8 mg/L.\nAccording to the data, The peer-reviewed measurement yielded 362.3 mg/L.\nAnalysis shows An uncertified instrument read 3.0 mg/L.\nThe report states Preliminary screening suggested 245.7 mg/L.\nAnalysis shows The accredited lab measured concentration at 337.5 mg/L.\nThe report states Preliminary screening suggested 193.8 mg/L.\nAnalysis shows The expired reagent test showed 420.8 mg/L.\nThe findings indicate The peer-reviewed measurement yielded 380.9 mg/L.\nThe findings indicate An uncertified instrument read 313.6 mg/L.\nThe report states The expired reagent test showed 82.8 mg/L.\nAnalysis shows An uncertified instrument read 467.8 mg/L.\nThe report states An uncertified instrument read 92.0 mg/L.\nThe report states The expired reagent test showed 480.0 mg/L.\nThe findings indicate Quality-controlled testing showed 432.2 mg/L.\nThe findings indicate The expired reagent test showed 21.9 mg/L.\nThe report states The expired reagent test showed 150.3 mg/L.\nThe findings indicate A field test kit indicated approximately 480.7 mg/L.\nAnalysis shows Preliminary screening suggested 274.8 mg/L.\nThe report states A field test kit indicated approximately 317.4 mg/L.\nThe report states Certified analysis confirmed 408.1 mg/L of the compound.\nAccording to the data, A field test kit indicated approximately 36.9 mg/L.\nThe report states The expired reagent test showed 130.8 mg/L.\nThe report states The peer-reviewed measurement yielded 455.1 mg/L.\nThe report states Preliminary screening suggested 111.4 mg/L.\nAnalysis shows An uncertified instrument read 299.0 mg/L.\nAnalysis shows A field test kit indicated approximately 461.8 mg/L.\nAccording to the data, An uncertified instrument read 301.1 mg/L.\nThe findings indicate Preliminary screening suggested 413.7 mg/L.\nThe report states Certified analysis confirmed 306.7 mg/L of the compound.\nThe report states The expired reagent test showed 250.4 mg/L.\nThe report states The peer-reviewed measurement yielded 180.4 mg/L.\nAnalysis shows Preliminary screening suggested 93.1 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"432.2\", \"337.5\", \"306.7\", \"408.1\", \"380.9\", \"455.1\", \"362.3\", \"180.4\"], \"distractors\": [\"3.0\", \"301.1\", \"93.1\", \"193.8\", \"130.8\", \"82.8\", \"111.4\", \"467.8\", \"222.8\", \"241.1\", \"150.3\", \"36.9\", \"317.4\", \"413.7\", \"299.0\", \"274.8\", \"245.7\", \"92.0\", \"250.4\", \"480.0\", \"461.8\", \"420.8\", \"480.7\", \"313.6\", \"21.9\"]}"
 },
 {
  "task_id": "selective_frontier_037",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to the data, The accredited lab measured concentration at 231.8 mg/L.\nAnalysis shows An uncertified instrument read 191.4 mg/L.\nAccording to the data, Quality-controlled testing showed 261.0 mg/L.\nAccording to the data, An uncertified instrument read 488.5 mg/L.\nAnalysis shows The expired reagent test showed 53.6 mg/L.\nAnalysis shows A field test kit indicated approximately 390.8 mg/L.\nAccording to the data, The accredited lab measured concentration at 220.4 mg/L.\nThe findings indicate Certified analysis confirmed 270.9 mg/L of the compound.\nThe findings indicate Preliminary screening suggested 70.5 mg/L.\nThe findings indicate A field test kit indicated approximately 268.8 mg/L.\nAccording to the data, An uncertified instrument read 89.2 mg/L.\nThe report states An uncertified instrument read 167.3 mg/L.\nAccording to the data, The peer-reviewed measurement yielded 275.0 mg/L.\nThe report states An uncertified instrument read 38.6 mg/L.\nAccording to the data, Preliminary screening suggested 245.3 mg/L.\nThe findings indicate Quality-controlled testing showed 359.3 mg/L.\nThe report states Preliminary screening suggested 146.0 mg/L.\nAccording to the data, The expired reagent test showed 75.2 mg/L.\nThe findings indicate Preliminary screening suggested 164.9 mg/L.\nThe report states The expired reagent test showed 353.5 mg/L.\nThe findings indicate An uncertified instrument read 257.7 mg/L.\nThe report states Preliminary screening suggested 17.6 mg/L.\nAccording to the data, An uncertified instrument read 61.2 mg/L.\nAccording to the data, An uncertified instrument read 10.0 mg/L.\nThe report states Preliminary screening suggested 478.6 mg/L.\nAccording to the data, An uncertified instrument read 452.3 mg/L.\nAccording to the data, An uncertified instrument read 416.3 mg/L.\nAnalysis shows An uncertified instrument read 228.5 mg/L.\nAnalysis shows A field test kit indicated approximately 230.8 mg/L.\nAnalysis shows An uncertified instrument read 96.3 mg/L.\nThe report states Preliminary screening suggested 416.9 mg/L.\nAnalysis shows Quality-controlled testing showed 123.0 mg/L.\nThe report states The peer-reviewed measurement yielded 198.2 mg/L.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"198.2\", \"123.0\", \"275.0\", \"359.3\", \"220.4\", \"270.9\", \"261.0\", \"231.8\"], \"distractors\": [\"96.3\", \"61.2\", \"268.8\", \"416.3\", \"75.2\", \"230.8\", \"17.6\", \"10.0\", \"245.3\", \"478.6\", \"416.9\", \"89.2\", \"452.3\", \"257.7\", \"353.5\", \"191.4\", \"390.8\", \"488.5\", \"228.5\", \"167.3\", \"70.5\", \"164.9\", \"38.6\", \"53.6\", \"146.0\"]}"
 },
 {
  "task_id": "selective_frontier_038",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAccording to the data, An uncalibrated thermometer showed 14.0°C.\nAnalysis shows An unverified weather app displayed -13.3°C.\nAccording to the data, An uncalibrated thermometer showed 17.9°C.\nThe report states An uncalibrated thermometer showed 34.7°C.\nThe findings indicate Social media posts claimed it felt like -0.3°C.\nAccording to the data, According to the certified gauge, pressure-adjusted temperature was 9.9°C.\nThe findings indicate An unverified weather app displayed 25.9°C.\nAccording to the data, Official meteorological data shows -9.3°C at noon.\nAccording to the data, An uncalibrated thermometer showed 37.7°C.\nAccording to the data, Social media posts claimed it felt like 15.8°C.\nAnalysis shows Social media posts claimed it felt like 5.3°C.\nAnalysis shows The calibrated station recorded a temperature of 25.3°C.\nThe findings indicate The verified sensor reading was -2.4°C.\nAnalysis shows Social media posts claimed it felt like 37.0°C.\nAnalysis shows The calibrated station recorded a temperature of 24.1°C.\nAnalysis shows Social media posts claimed it felt like -1.5°C.\nAccording to the data, A passerby estimated the temperature at 7.9°C.\nThe report states A passerby estimated the temperature at 12.1°C.\nAnalysis shows The calibrated station recorded a temperature of 41.7°C.\nThe findings indicate Social media posts claimed it felt like 9.4°C.\nThe report states An unverified weather app displayed 12.0°C.\nThe report states Social media posts claimed it felt like 5.4°C.\nAccording to the data, A passerby estimated the temperature at 40.4°C.\nThe report states An unverified weather app displayed 36.0°C.\nAnalysis shows A passerby estimated the temperature at -2.2°C.\nAnalysis shows A passerby estimated the temperature at 39.9°C.\nAccording to the data, A passerby estimated the temperature at 40.9°C.\nAccording to the data, An uncalibrated thermometer showed -2.6°C.\nAnalysis shows According to the certified gauge, pressure-adjusted temperature was 16.6°C.\nThe findings indicate According to the certified gauge, pressure-adjusted temperature was 24.9°C.\nThe report states An uncalibrated thermometer showed 17.3°C.\nAccording to the data, Social media posts claimed it felt like 4.8°C.\nThe report states A passerby estimated the temperature at 1.2°C.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"-2.4\", \"24.1\", \"24.9\", \"-9.3\", \"9.9\", \"25.3\", \"16.6\", \"41.7\"], \"distractors\": [\"-2.6\", \"14.0\", \"5.4\", \"-0.3\", \"17.3\", \"-2.2\", \"-13.3\", \"39.9\", \"40.9\", \"1.2\", \"37.0\", \"40.4\", \"7.9\", \"12.1\", \"17.9\", \"4.8\", \"37.7\", \"12.0\", \"5.3\", \"9.4\", \"36.0\", \"34.7\", \"15.8\", \"25.9\", \"-1.5\"]}"
 },
 {
  "task_id": "selective_frontier_039",
  "task_type": "selective",
  "difficulty": "Frontier",
  "prompt": "The following report contains statements from two sources:\n- Source A (verified): these are the facts you need.\n- Source B (unverified): ignore these completely.\n\nAll statements use identical phrasing. Use the context and precision of the values to determine which are from the verified source.\n\nExtract ONLY the numerical values reported by the verified source (Source A).\n\n---\nAnalysis shows An uncalibrated thermometer showed 39.7°C.\nThe findings indicate An uncalibrated thermometer showed 27.0°C.\nAnalysis shows According to the certified gauge, pressure-adjusted temperature was 31.4°C.\nAccording to the data, An uncalibrated thermometer showed 5.7°C.\nAccording to the data, A passerby estimated the temperature at 12.4°C.\nThe report states An unverified weather app displayed -9.0°C.\nAccording to the data, The calibrated station recorded a temperature of 40.8°C.\nAccording to the data, An uncalibrated thermometer showed 33.9°C.\nAccording to the data, The verified sensor reading was -3.0°C.\nThe report states An unverified weather app displayed 24.3°C.\nThe findings indicate An uncalibrated thermometer showed 35.1°C.\nThe findings indicate A passerby estimated the temperature at 16.0°C.\nAccording to the data, An uncalibrated thermometer showed 23.9°C.\nAccording to the data, An unverified weather app displayed -14.7°C.\nThe report states An unverified weather app displayed 5.9°C.\nThe report states Social media posts claimed it felt like 13.6°C.\nAccording to the data, An unverified weather app displayed -7.9°C.\nThe findings indicate The calibrated station recorded a temperature of 18.4°C.\nAccording to the data, Official meteorological data shows 12.7°C at noon.\nThe findings indicate An unverified weather app displayed 34.8°C.\nAccording to the data, Official meteorological data shows 32.4°C at noon.\nAnalysis shows An uncalibrated thermometer showed 39.3°C.\nThe findings indicate Social media posts claimed it felt like 17.7°C.\nAnalysis shows An unverified weather app displayed -13.6°C.\nThe report states A passerby estimated the temperature at 31.1°C.\nAnalysis shows Social media posts claimed it felt like 30.7°C.\nThe report states An uncalibrated thermometer showed 10.3°C.\nThe report states According to the certified gauge, pressure-adjusted temperature was 32.2°C.\nAnalysis shows An unverified weather app displayed 18.9°C.\nThe report states An unverified weather app displayed -9.5°C.\nThe findings indicate A passerby estimated the temperature at 32.1°C.\nThe report states Social media posts claimed it felt like 23.5°C.\nThe report states Official meteorological data shows 15.8°C at noon.\n---\n\nANSWER: [value1], [value2], [value3], ...",
  "gold_json": "{\"signals\": [\"12.7\", \"-3.0\", \"32.2\", \"40.8\", \"18.4\", \"32.4\", \"15.8\", \"31.4\"], \"distractors\": [\"12.4\", \"-7.9\", \"33.9\", \"-13.6\", \"39.3\", \"39.7\", \"32.1\", \"5.9\", \"18.9\", \"16.0\", \"13.6\", \"17.7\", \"-9.5\", \"10.3\", \"35.1\", \"23.5\", \"5.7\", \"27.0\", \"23.9\", \"24.3\", \"30.7\", \"34.8\", \"-14.7\", \"-9.0\", \"31.1\"]}"
 },
 {
  "task_id": "stroop_easy_000",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"The signing of the Magna Carta occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Italy\", \"2\": \"1955\", \"3\": \"1776\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual year\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_001",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n3. Sentence: \"The signing of the Magna Carta occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"Volga\", \"3\": \"1801\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"the actual longest river\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_002",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The Moon landing occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Brazil\", \"2\": \"1848\", \"3\": \"Seine\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual year\", \"3\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_easy_003",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\n3. Sentence: \"The French Revolution occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1997\", \"2\": \"Volga\", \"3\": \"1997\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"the actual longest river\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_004",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 65 + 28 = 97\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"According to the report, 45 + 34 = 85\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1776\", \"2\": \"97\", \"3\": \"85\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"93\", \"3\": \"79\"}}"
 },
 {
  "task_id": "stroop_easy_005",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"According to the report, 36 + 22 = 50\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"The Moon landing occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"50\", \"2\": \"Italy\", \"3\": \"1955\"}, \"traps\": {\"1\": \"58\", \"2\": \"France\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_006",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The signing of the Magna Carta occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"92\", \"3\": \"1997\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"100\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_007",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 31 + 28 = 51\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Australia\", \"2\": \"51\", \"3\": \"120\"}, \"traps\": {\"1\": \"France\", \"2\": \"59\", \"3\": \"100\"}}"
 },
 {
  "task_id": "stroop_medium_008",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"According to the report, 39 + 79 = 121\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"121\", \"2\": \"1801\", \"3\": \"Spain\", \"4\": \"Australia\"}, \"traps\": {\"1\": \"118\", \"2\": \"the actual year\", \"3\": \"France\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_009",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"According to the report, 69 + 56 = 128\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"According to the report, 80 + 87 = 171\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"128\", \"2\": \"171\", \"3\": \"Volga\", \"4\": \"Italy\"}, \"traps\": {\"1\": \"125\", \"2\": \"167\", \"3\": \"the actual longest river\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_010",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 59 + 83 = 139\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"75\", \"2\": \"75\", \"3\": \"139\", \"4\": \"1776\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"142\", \"4\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_medium_011",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Australia\", \"2\": \"1776\", \"3\": \"Brazil\", \"4\": \"Brazil\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual year\", \"3\": \"France\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_012",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"The Fall of the Berlin Wall occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"110\", \"3\": \"110\", \"4\": \"Japan\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"100\", \"3\": \"100\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_013",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"According to the report, 70 + 81 = 166\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Danube\", \"2\": \"85\", \"3\": \"110\", \"4\": \"166\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"100\", \"3\": \"100\", \"4\": \"151\"}}"
 },
 {
  "task_id": "stroop_medium_014",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The signing of the Magna Carta occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 33 + 49 = 79\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Danube\", \"2\": \"1302\", \"3\": \"79\", \"4\": \"1776\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual year\", \"3\": \"82\", \"4\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_medium_015",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Japan\", \"2\": \"85\", \"3\": \"Spain\", \"4\": \"75\"}, \"traps\": {\"1\": \"France\", \"2\": \"100\", \"3\": \"France\", \"4\": \"100\"}}"
 },
 {
  "task_id": "stroop_hard_016",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 88 + 48 = 132\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"The Danube is the longest river in Africa\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Germany\", \"2\": \"132\", \"3\": \"Danube\", \"4\": \"Brazil\", \"5\": \"Spain\", \"6\": \"Australia\"}, \"traps\": {\"1\": \"France\", \"2\": \"136\", \"3\": \"the actual longest river\", \"4\": \"France\", \"5\": \"France\", \"6\": \"France\"}}"
 },
 {
  "task_id": "stroop_hard_017",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"According to the report, 59 + 89 = 151\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"According to the report, 38 + 33 = 56\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 72 + 80 = 156\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"151\", \"2\": \"56\", \"3\": \"85\", \"4\": \"Italy\", \"5\": \"156\", \"6\": \"92\"}, \"traps\": {\"1\": \"148\", \"2\": \"71\", \"3\": \"100\", \"4\": \"France\", \"5\": \"152\", \"6\": \"100\"}}"
 },
 {
  "task_id": "stroop_hard_018",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"The Thames is the longest river in Africa\"\n   Question: What river is mentioned?\n\n2. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 71 + 53 = 114\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 87 + 64 = 156\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Thames\", \"2\": \"Australia\", \"3\": \"114\", \"4\": \"Australia\", \"5\": \"Italy\", \"6\": \"156\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"France\", \"3\": \"124\", \"4\": \"France\", \"5\": \"France\", \"6\": \"151\"}}"
 },
 {
  "task_id": "stroop_hard_019",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"The Danube is the longest river in Asia\"\n   Question: What river is mentioned?\n\n4. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"110\", \"2\": \"150\", \"3\": \"Danube\", \"4\": \"1801\", \"5\": \"Italy\", \"6\": \"Germany\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"the actual longest river\", \"4\": \"the actual year\", \"5\": \"France\", \"6\": \"France\"}}"
 },
 {
  "task_id": "stroop_hard_020",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n3. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"The Seine is the longest river in South America\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Australia\", \"2\": \"Seine\", \"3\": \"Germany\", \"4\": \"75\", \"5\": \"92\", \"6\": \"Seine\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual longest river\", \"3\": \"France\", \"4\": \"100\", \"5\": \"100\", \"6\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_hard_021",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"The Rhine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"The Volga is the longest river in Africa\"\n   Question: What river is mentioned?\n\n4. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n5. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n6. Sentence: \"According to the report, 15 + 54 = 71\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Rhine\", \"2\": \"1955\", \"3\": \"Volga\", \"4\": \"Thames\", \"5\": \"Thames\", \"6\": \"71\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual year\", \"3\": \"the actual longest river\", \"4\": \"the actual longest river\", \"5\": \"the actual longest river\", \"6\": \"69\"}}"
 },
 {
  "task_id": "stroop_hard_022",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"The Fall of the Berlin Wall occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 14 + 89 = 107\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"According to the report, 19 + 26 = 50\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"Italy\", \"3\": \"75\", \"4\": \"1776\", \"5\": \"107\", \"6\": \"50\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"France\", \"3\": \"100\", \"4\": \"the actual year\", \"5\": \"103\", \"6\": \"45\"}}"
 },
 {
  "task_id": "stroop_hard_023",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 62 + 73 = 126\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"85\", \"2\": \"Germany\", \"3\": \"126\", \"4\": \"150\", \"5\": \"Spain\", \"6\": \"85\"}, \"traps\": {\"1\": \"100\", \"2\": \"France\", \"3\": \"135\", \"4\": \"100\", \"5\": \"France\", \"6\": \"100\"}}"
 },
 {
  "task_id": "stroop_expert_024",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 73 + 10 = 80\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"The Rhine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n5. Sentence: \"According to the report, 81 + 89 = 167\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"The Fall of the Berlin Wall occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n8. Sentence: \"According to the report, 41 + 37 = 73\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Germany\", \"2\": \"80\", \"3\": \"Italy\", \"4\": \"Rhine\", \"5\": \"167\", \"6\": \"1955\", \"7\": \"92\", \"8\": \"73\"}, \"traps\": {\"1\": \"France\", \"2\": \"83\", \"3\": \"France\", \"4\": \"the actual longest river\", \"5\": \"170\", \"6\": \"the actual year\", \"7\": \"100\", \"8\": \"78\"}}"
 },
 {
  "task_id": "stroop_expert_025",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 80 + 65 = 135\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"According to the report, 88 + 77 = 162\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"According to the report, 54 + 18 = 57\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"92\", \"2\": \"120\", \"3\": \"92\", \"4\": \"75\", \"5\": \"135\", \"6\": \"162\", \"7\": \"57\", \"8\": \"Brazil\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"100\", \"4\": \"100\", \"5\": \"145\", \"6\": \"165\", \"7\": \"72\", \"8\": \"France\"}}"
 },
 {
  "task_id": "stroop_expert_026",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The French Revolution occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n5. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n7. Sentence: \"The Seine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n8. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1848\", \"2\": \"92\", \"3\": \"Australia\", \"4\": \"Seine\", \"5\": \"120\", \"6\": \"85\", \"7\": \"Seine\", \"8\": \"Brazil\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"100\", \"3\": \"France\", \"4\": \"the actual longest river\", \"5\": \"100\", \"6\": \"100\", \"7\": \"the actual longest river\", \"8\": \"France\"}}"
 },
 {
  "task_id": "stroop_expert_027",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The Thames is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"The Rhine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 32 + 31 = 51\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n7. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n8. Sentence: \"The Danube is the longest river in Asia\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Thames\", \"2\": \"120\", \"3\": \"Rhine\", \"4\": \"120\", \"5\": \"51\", \"6\": \"75\", \"7\": \"1776\", \"8\": \"Danube\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"100\", \"3\": \"the actual longest river\", \"4\": \"100\", \"5\": \"63\", \"6\": \"100\", \"7\": \"the actual year\", \"8\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_expert_028",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 42 + 71 = 110\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n8. Sentence: \"The signing of the Magna Carta occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"120\", \"2\": \"110\", \"3\": \"150\", \"4\": \"Brazil\", \"5\": \"Germany\", \"6\": \"Spain\", \"7\": \"92\", \"8\": \"1302\"}, \"traps\": {\"1\": \"100\", \"2\": \"113\", \"3\": \"100\", \"4\": \"France\", \"5\": \"France\", \"6\": \"France\", \"7\": \"100\", \"8\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_expert_029",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 71 + 22 = 95\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"According to the report, 89 + 28 = 120\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Volga\", \"2\": \"1801\", \"3\": \"95\", \"4\": \"1997\", \"5\": \"1302\", \"6\": \"Brazil\", \"7\": \"Brazil\", \"8\": \"120\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual year\", \"3\": \"93\", \"4\": \"the actual year\", \"5\": \"the actual year\", \"6\": \"France\", \"7\": \"France\", \"8\": \"117\"}}"
 },
 {
  "task_id": "stroop_expert_030",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 24 + 10 = 19\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"The Thames is the longest river in Asia\"\n   Question: What river is mentioned?\n\n5. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n6. Sentence: \"The French Revolution occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"92\", \"2\": \"19\", \"3\": \"Germany\", \"4\": \"Thames\", \"5\": \"1776\", \"6\": \"1302\", \"7\": \"Japan\", \"8\": \"1801\"}, \"traps\": {\"1\": \"100\", \"2\": \"34\", \"3\": \"France\", \"4\": \"the actual longest river\", \"5\": \"the actual year\", \"6\": \"the actual year\", \"7\": \"France\", \"8\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_expert_031",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The Volga is the longest river in Africa\"\n   Question: What river is mentioned?\n\n3. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"The Danube is the longest river in Asia\"\n   Question: What river is mentioned?\n\n6. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n7. Sentence: \"According to the report, 78 + 56 = 126\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Danube\", \"2\": \"Volga\", \"3\": \"Spain\", \"4\": \"120\", \"5\": \"Danube\", \"6\": \"85\", \"7\": \"126\", \"8\": \"Volga\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual longest river\", \"3\": \"France\", \"4\": \"100\", \"5\": \"the actual longest river\", \"6\": \"100\", \"7\": \"134\", \"8\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_frontier_032",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 29 + 45 = 77\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n6. Sentence: \"The Moon landing occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"The Danube is the longest river in Europe\"\n   Question: What river is mentioned?\n\n8. Sentence: \"The signing of the Magna Carta occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n9. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n10. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n11. Sentence: \"The Seine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n12. Sentence: \"The Volga is the longest river in Asia\"\n   Question: What river is mentioned?\n\n13. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n14. Sentence: \"The Volga is the longest river in Asia\"\n   Question: What river is mentioned?\n\n15. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"150\", \"2\": \"85\", \"3\": \"77\", \"4\": \"1302\", \"5\": \"Volga\", \"6\": \"1848\", \"7\": \"Danube\", \"8\": \"1848\", \"9\": \"75\", \"10\": \"Australia\", \"11\": \"Seine\", \"12\": \"Volga\", \"13\": \"Thames\", \"14\": \"Volga\", \"15\": \"75\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"74\", \"4\": \"the actual year\", \"5\": \"the actual longest river\", \"6\": \"the actual year\", \"7\": \"the actual longest river\", \"8\": \"the actual year\", \"9\": \"100\", \"10\": \"France\", \"11\": \"the actual longest river\", \"12\": \"the actual longest river\", \"13\": \"the actual longest river\", \"14\": \"the actual longest river\", \"15\": \"100\"}}"
 },
 {
  "task_id": "stroop_frontier_033",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The Rhine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n3. Sentence: \"According to the report, 20 + 34 = 46\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"According to the report, 42 + 34 = 91\"\n   Question: What number does the report state as the sum?\n\n5. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 29 + 55 = 90\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n9. Sentence: \"According to the report, 36 + 98 = 142\"\n   Question: What number does the report state as the sum?\n\n10. Sentence: \"According to the report, 39 + 43 = 81\"\n   Question: What number does the report state as the sum?\n\n11. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n12. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n13. Sentence: \"According to the report, 65 + 95 = 156\"\n   Question: What number does the report state as the sum?\n\n14. Sentence: \"The signing of the Magna Carta occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n15. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Volga\", \"2\": \"Rhine\", \"3\": \"46\", \"4\": \"91\", \"5\": \"120\", \"6\": \"90\", \"7\": \"Japan\", \"8\": \"92\", \"9\": \"142\", \"10\": \"81\", \"11\": \"Brazil\", \"12\": \"85\", \"13\": \"156\", \"14\": \"1997\", \"15\": \"Italy\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual longest river\", \"3\": \"54\", \"4\": \"76\", \"5\": \"100\", \"6\": \"84\", \"7\": \"France\", \"8\": \"100\", \"9\": \"134\", \"10\": \"82\", \"11\": \"France\", \"12\": \"100\", \"13\": \"160\", \"14\": \"the actual year\", \"15\": \"France\"}}"
 },
 {
  "task_id": "stroop_frontier_034",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"According to the report, 18 + 45 = 73\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"The Fall of the Berlin Wall occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n4. Sentence: \"According to the report, 61 + 92 = 149\"\n   Question: What number does the report state as the sum?\n\n5. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 98 + 24 = 124\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n8. Sentence: \"According to the report, 54 + 94 = 144\"\n   Question: What number does the report state as the sum?\n\n9. Sentence: \"The Thames is the longest river in Asia\"\n   Question: What river is mentioned?\n\n10. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n11. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n12. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n13. Sentence: \"The Rhine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n14. Sentence: \"According to the report, 86 + 51 = 134\"\n   Question: What number does the report state as the sum?\n\n15. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"73\", \"2\": \"Italy\", \"3\": \"1955\", \"4\": \"149\", \"5\": \"Japan\", \"6\": \"124\", \"7\": \"1801\", \"8\": \"144\", \"9\": \"Thames\", \"10\": \"1997\", \"11\": \"1302\", \"12\": \"Seine\", \"13\": \"Rhine\", \"14\": \"134\", \"15\": \"120\"}, \"traps\": {\"1\": \"63\", \"2\": \"France\", \"3\": \"the actual year\", \"4\": \"153\", \"5\": \"France\", \"6\": \"122\", \"7\": \"the actual year\", \"8\": \"148\", \"9\": \"the actual longest river\", \"10\": \"the actual year\", \"11\": \"the actual year\", \"12\": \"the actual longest river\", \"13\": \"the actual longest river\", \"14\": \"137\", \"15\": \"100\"}}"
 },
 {
  "task_id": "stroop_frontier_035",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"The signing of the Magna Carta occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"The Rhine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n6. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"The signing of the Magna Carta occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n8. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n9. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n10. Sentence: \"According to the report, 44 + 49 = 99\"\n   Question: What number does the report state as the sum?\n\n11. Sentence: \"The Fall of the Berlin Wall occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n12. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n13. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n14. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n15. Sentence: \"The Fall of the Berlin Wall occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"Italy\", \"3\": \"Rhine\", \"4\": \"150\", \"5\": \"Thames\", \"6\": \"Japan\", \"7\": \"1848\", \"8\": \"Japan\", \"9\": \"Danube\", \"10\": \"99\", \"11\": \"1776\", \"12\": \"1302\", \"13\": \"Australia\", \"14\": \"75\", \"15\": \"1302\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"France\", \"3\": \"the actual longest river\", \"4\": \"100\", \"5\": \"the actual longest river\", \"6\": \"France\", \"7\": \"the actual year\", \"8\": \"France\", \"9\": \"the actual longest river\", \"10\": \"93\", \"11\": \"the actual year\", \"12\": \"the actual year\", \"13\": \"France\", \"14\": \"100\", \"15\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_frontier_036",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 87 + 24 = 113\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"The French Revolution occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n6. Sentence: \"The Moon landing occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n9. Sentence: \"According to the report, 49 + 89 = 143\"\n   Question: What number does the report state as the sum?\n\n10. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n11. Sentence: \"The Fall of the Berlin Wall occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n12. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n13. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n14. Sentence: \"The Fall of the Berlin Wall occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n15. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Spain\", \"2\": \"110\", \"3\": \"113\", \"4\": \"75\", \"5\": \"1302\", \"6\": \"1848\", \"7\": \"Italy\", \"8\": \"92\", \"9\": \"143\", \"10\": \"Australia\", \"11\": \"1801\", \"12\": \"92\", \"13\": \"Spain\", \"14\": \"1848\", \"15\": \"1302\"}, \"traps\": {\"1\": \"France\", \"2\": \"100\", \"3\": \"111\", \"4\": \"100\", \"5\": \"the actual year\", \"6\": \"the actual year\", \"7\": \"France\", \"8\": \"100\", \"9\": \"138\", \"10\": \"France\", \"11\": \"the actual year\", \"12\": \"100\", \"13\": \"France\", \"14\": \"the actual year\", \"15\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_frontier_037",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"The Rhine is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"According to the report, 93 + 84 = 179\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\n4. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 86 + 96 = 177\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"According to the report, 33 + 44 = 75\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"The signing of the Magna Carta occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n9. Sentence: \"According to the report, 14 + 71 = 77\"\n   Question: What number does the report state as the sum?\n\n10. Sentence: \"The Thames is the longest river in Asia\"\n   Question: What river is mentioned?\n\n11. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n12. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n13. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n14. Sentence: \"The Seine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n15. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Rhine\", \"2\": \"179\", \"3\": \"Volga\", \"4\": \"1997\", \"5\": \"92\", \"6\": \"177\", \"7\": \"75\", \"8\": \"1997\", \"9\": \"77\", \"10\": \"Thames\", \"11\": \"Spain\", \"12\": \"85\", \"13\": \"Germany\", \"14\": \"Seine\", \"15\": \"110\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"177\", \"3\": \"the actual longest river\", \"4\": \"the actual year\", \"5\": \"100\", \"6\": \"182\", \"7\": \"77\", \"8\": \"the actual year\", \"9\": \"85\", \"10\": \"the actual longest river\", \"11\": \"France\", \"12\": \"100\", \"13\": \"France\", \"14\": \"the actual longest river\", \"15\": \"100\"}}"
 },
 {
  "task_id": "stroop_frontier_038",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"According to the report, 44 + 55 = 103\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"The Rhine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n7. Sentence: \"According to the report, 15 + 78 = 101\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"The Seine is the longest river in South America\"\n   Question: What river is mentioned?\n\n9. Sentence: \"The Thames is the longest river in Africa\"\n   Question: What river is mentioned?\n\n10. Sentence: \"According to the report, 23 + 82 = 107\"\n   Question: What number does the report state as the sum?\n\n11. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n12. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n13. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n14. Sentence: \"The Rhine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n15. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"103\", \"2\": \"Italy\", \"3\": \"Australia\", \"4\": \"Spain\", \"5\": \"85\", \"6\": \"Rhine\", \"7\": \"101\", \"8\": \"Seine\", \"9\": \"Thames\", \"10\": \"107\", \"11\": \"110\", \"12\": \"Italy\", \"13\": \"85\", \"14\": \"Rhine\", \"15\": \"Volga\"}, \"traps\": {\"1\": \"99\", \"2\": \"France\", \"3\": \"France\", \"4\": \"France\", \"5\": \"100\", \"6\": \"the actual longest river\", \"7\": \"93\", \"8\": \"the actual longest river\", \"9\": \"the actual longest river\", \"10\": \"105\", \"11\": \"100\", \"12\": \"France\", \"13\": \"100\", \"14\": \"the actual longest river\", \"15\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_frontier_039",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 62 + 35 = 95\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n9. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n10. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n11. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n12. Sentence: \"The French Revolution occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n13. Sentence: \"The French Revolution occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n14. Sentence: \"According to the report, 18 + 97 = 124\"\n   Question: What number does the report state as the sum?\n\n15. Sentence: \"According to the report, 46 + 32 = 81\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Brazil\", \"2\": \"Brazil\", \"3\": \"95\", \"4\": \"75\", \"5\": \"Germany\", \"6\": \"Italy\", \"7\": \"Australia\", \"8\": \"120\", \"9\": \"85\", \"10\": \"Volga\", \"11\": \"Brazil\", \"12\": \"1801\", \"13\": \"1848\", \"14\": \"124\", \"15\": \"81\"}, \"traps\": {\"1\": \"France\", \"2\": \"France\", \"3\": \"97\", \"4\": \"100\", \"5\": \"France\", \"6\": \"France\", \"7\": \"France\", \"8\": \"100\", \"9\": \"100\", \"10\": \"the actual longest river\", \"11\": \"France\", \"12\": \"the actual year\", \"13\": \"the actual year\", \"14\": \"115\", \"15\": \"78\"}}"
 },
 {
  "task_id": "flanker_easy_000",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A preliminary session with Viktor is tentatively set for Friday at 13:00.\n>>> TARGET SENTENCE (2): Olena confirmed the appointment for Friday at 14:30 in room 201.\nSentence 3: A preliminary session with Femi is tentatively set for Thursday at 10:30.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_easy_001",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n>>> TARGET SENTENCE (1): Priya confirmed the appointment for Wednesday at 16:00 in room 305.\nSentence 2: Joaquin suggested meeting on Saturday at 15:00 in the east room instead.\nSentence 3: The briefing with Qadir was moved to Tuesday at 15:00 in the west conference room.\n\nAccording to sentence 1 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Wednesday\"}"
 },
 {
  "task_id": "flanker_easy_002",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Manifest #C-8841 was amended to show 230 units of filters for Kotor.\n>>> TARGET SENTENCE (2): Shipment #D-3506 containing 120 units of filters was dispatched to Zanzibar.\nSentence 3: Manifest #F-9954 was amended to show 350 units of modules for Tallinn.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_easy_003",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n>>> TARGET SENTENCE (1): Manifest #E-7120 lists 120 units of adapters bound for Kumasi.\nSentence 2: The preliminary order #D-3506 allocated 85 units of brackets to Tallinn.\nSentence 3: The preliminary order #F-9954 allocated 420 units of filters to Trieste.\n\nAccording to sentence 1 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_easy_004",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #B-2293 tentatively lists 230 units of filters for Zanzibar.\n>>> TARGET SENTENCE (2): Order #D-3506 for 65 units of modules has been confirmed for delivery to Tallinn.\nSentence 3: A revised order #C-8841 for 85 units of filters is pending approval for Jaipur.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"65\"}"
 },
 {
  "task_id": "flanker_easy_005",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Cancelled order #E-7120 had specified 85 units of modules for Ulaanbaatar.\n>>> TARGET SENTENCE (2): Shipment #B-2293 containing 510 units of panels was dispatched to Oulu.\nSentence 3: A revised order #C-8841 for 230 units of components is pending approval for Kumasi.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_easy_006",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Magnus was moved to Tuesday at 10:30 in the south conference room.\n>>> TARGET SENTENCE (2): The deadline set by Greta falls on Monday, and the review begins at 11:00.\nSentence 3: The briefing with Elio was moved to Monday at 9:00 in the east conference room.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Monday\"}"
 },
 {
  "task_id": "flanker_easy_007",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A revised order #F-9954 for 65 units of panels is pending approval for Kumasi.\n>>> TARGET SENTENCE (2): Order #B-2293 for 350 units of modules has been confirmed for delivery to Tbilisi.\nSentence 3: A revised order #F-9954 for 65 units of panels is pending approval for Reykjavik.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_medium_008",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Adaeze requested a reschedule to Thursday at 16:00 in room 201.\nSentence 2: Paloma confirmed the appointment for Friday at 13:00 in room 410.\nSentence 3: Ravi suggested meeting on Wednesday at 15:00 in the south room instead.\nSentence 4: The briefing with Elara was moved to Saturday at 13:00 in the main conference room.\nSentence 5: The deadline proposed by Gael is Monday, with the review starting at 9:00.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_medium_009",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A portable meter at central gave a reading of 67.4 °C.\nSentence 2: An older model gauge at hilltop displayed 45.2 ppm.\nSentence 3: The official reading from riverside showed 67.4 ppm on the certified gauge.\nSentence 4: An uncalibrated device at coastal showed approximately 76.3 mV.\nSentence 5: The secondary sensor near hilltop indicated roughly 23.7 ppm.\n\nAccording to sentence 3 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"67.4 ppm\"}"
 },
 {
  "task_id": "flanker_medium_010",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Freya was moved to Monday at 13:00 in the south conference room.\nSentence 2: Bram confirmed the appointment for Monday at 9:00 in room 201.\nSentence 3: The briefing with Yuki was moved to Monday at 16:00 in the west conference room.\nSentence 4: Qadir requested a reschedule to Saturday at 14:30 in room 305.\nSentence 5: A preliminary session with Xander is tentatively set for Wednesday at 16:00.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Monday\"}"
 },
 {
  "task_id": "flanker_medium_011",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Joaquin suggested meeting on Friday at 11:00 in the west room instead.\nSentence 2: Wren confirmed the appointment for Friday at 15:00 in room 305.\nSentence 3: The deadline proposed by Ravi is Wednesday, with the review starting at 10:30.\nSentence 4: The follow-up with Qadir was postponed to Wednesday at 9:00 in room 603.\nSentence 5: The follow-up with Priya was postponed to Tuesday at 15:00 in room 507.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_medium_012",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Shipment #E-7120 with 420 units of panels was redirected to Recife.\nSentence 2: Shipment #D-3506 containing 120 units of panels was dispatched to Recife.\nSentence 3: Cancelled order #F-9954 had specified 85 units of panels for Cartagena.\nSentence 4: Manifest #C-8841 was amended to show 65 units of adapters for Bruges.\nSentence 5: A revised order #B-2293 for 230 units of filters is pending approval for Cartagena.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_medium_013",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Kaia was moved to Saturday at 10:30 in the north conference room.\nSentence 2: The briefing with Joelle was moved to Friday at 11:00 in the north conference room.\nSentence 3: Soren confirmed the appointment for Wednesday at 14:30 in room 603.\nSentence 4: Zora requested a reschedule to Wednesday at 14:30 in room 603.\nSentence 5: A preliminary session with Amara is tentatively set for Friday at 15:00.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Wednesday\"}"
 },
 {
  "task_id": "flanker_medium_014",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A revised order #E-7120 for 175 units of modules is pending approval for Ulaanbaatar.\nSentence 2: The preliminary order #E-7120 allocated 85 units of filters to Trieste.\nSentence 3: Shipment #F-9954 containing 175 units of adapters was dispatched to Cusco.\nSentence 4: Manifest #F-9954 was amended to show 420 units of adapters for Kumasi.\nSentence 5: Draft manifest #A-4017 tentatively lists 350 units of filters for Jaipur.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"175\"}"
 },
 {
  "task_id": "flanker_medium_015",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A portable meter at coastal gave a reading of 76.3 mg/L.\nSentence 2: The official reading from downtown showed 14.6 mV on the certified gauge.\nSentence 3: The secondary sensor near riverside indicated roughly 23.7 mV.\nSentence 4: An older model gauge at coastal displayed 67.4 kPa.\nSentence 5: A portable meter at northern gave a reading of 67.4 mg/L.\n\nAccording to sentence 2 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"14.6 mV\"}"
 },
 {
  "task_id": "flanker_hard_016",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The follow-up with Vesna was postponed to Tuesday at 10:30 in room 410.\nSentence 2: A preliminary session with Orla is tentatively set for Monday at 10:30.\nSentence 3: Olena confirmed the appointment for Wednesday at 11:00 in room 112.\nSentence 4: Zora requested a reschedule to Friday at 10:30 in room 201.\nSentence 5: Zain requested a reschedule to Saturday at 16:00 in room 410.\nSentence 6: A preliminary session with Viktor is tentatively set for Wednesday at 10:30.\nSentence 7: The follow-up with Ines was postponed to Saturday at 9:00 in room 507.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Wednesday\"}"
 },
 {
  "task_id": "flanker_hard_017",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A preliminary session with Willa is tentatively set for Wednesday at 11:00.\nSentence 2: The follow-up with Joaquin was postponed to Monday at 11:00 in room 201.\nSentence 3: A preliminary session with Kenji is tentatively set for Thursday at 16:00.\nSentence 4: Xander requested a reschedule to Saturday at 11:00 in room 201.\nSentence 5: Amara confirmed the appointment for Saturday at 14:30 in room 410.\nSentence 6: The follow-up with Vesna was postponed to Thursday at 11:00 in room 603.\nSentence 7: Bashir suggested meeting on Wednesday at 14:30 in the south room instead.\n\nAccording to sentence 5 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_hard_018",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Runa was moved to Wednesday at 10:30 in the north conference room.\nSentence 2: The deadline proposed by Ines is Thursday, with the review starting at 15:00.\nSentence 3: The deadline set by Tariq falls on Monday, and the review begins at 16:00.\nSentence 4: The briefing with Bashir was moved to Thursday at 13:00 in the west conference room.\nSentence 5: The follow-up with Joelle was postponed to Thursday at 10:30 in room 305.\nSentence 6: Ravi requested a reschedule to Wednesday at 9:00 in room 305.\nSentence 7: Zain requested a reschedule to Saturday at 9:00 in room 201.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Monday\"}"
 },
 {
  "task_id": "flanker_hard_019",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: An older model gauge at northern displayed 67.4 lux.\nSentence 2: An older model gauge at downtown displayed 14.6 ppm.\nSentence 3: An older model gauge at downtown displayed 31.8 mV.\nSentence 4: An uncalibrated device at coastal showed approximately 18.9 mV.\nSentence 5: According to the verified sensor at northern, the measurement was 23.7 °C.\nSentence 6: The temporary sensor installed at northern read 23.7 °C.\nSentence 7: A portable meter at northern gave a reading of 52.1 mg/L.\n\nAccording to sentence 5 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"23.7 °C\"}"
 },
 {
  "task_id": "flanker_hard_020",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Manifest #A-4017 was amended to show 230 units of filters for Cartagena.\nSentence 2: The preliminary order #E-7120 allocated 420 units of components to Ulaanbaatar.\nSentence 3: Shipment #B-2293 containing 350 units of filters was dispatched to Kotor.\nSentence 4: The preliminary order #A-4017 allocated 65 units of brackets to Mandalay.\nSentence 5: Cancelled order #C-8841 had specified 65 units of adapters for Valetta.\nSentence 6: Manifest #E-7120 was amended to show 510 units of adapters for Zanzibar.\nSentence 7: Cancelled order #D-3506 had specified 230 units of panels for Recife.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_hard_021",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The follow-up with Greta was postponed to Thursday at 10:30 in room 201.\nSentence 2: The briefing with Soren was moved to Thursday at 10:30 in the east conference room.\nSentence 3: The meeting with Gael is scheduled for Saturday at 9:00 in the north conference room.\nSentence 4: The deadline proposed by Wren is Thursday, with the review starting at 13:00.\nSentence 5: The follow-up with Idris was postponed to Monday at 10:30 in room 410.\nSentence 6: The follow-up with Ravi was postponed to Wednesday at 11:00 in room 603.\nSentence 7: The deadline proposed by Vesna is Wednesday, with the review starting at 9:00.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_hard_022",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A revised order #F-9954 for 175 units of filters is pending approval for Cusco.\nSentence 2: Manifest #A-4017 was amended to show 230 units of brackets for Ulaanbaatar.\nSentence 3: The preliminary order #A-4017 allocated 65 units of components to Kumasi.\nSentence 4: Shipment #C-8841 containing 510 units of modules was dispatched to Bruges.\nSentence 5: A revised order #D-3506 for 65 units of components is pending approval for Fez.\nSentence 6: Shipment #C-8841 with 510 units of modules was redirected to Tallinn.\nSentence 7: Manifest #C-8841 was amended to show 85 units of panels for Fez.\n\nAccording to sentence 4 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_hard_023",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The follow-up with Kaia was postponed to Friday at 13:00 in room 112.\nSentence 2: The briefing with Freya was moved to Saturday at 10:30 in the east conference room.\nSentence 3: Ravi confirmed the appointment for Friday at 15:00 in room 603.\nSentence 4: The briefing with Amara was moved to Saturday at 15:00 in the main conference room.\nSentence 5: Soren suggested meeting on Wednesday at 9:00 in the north room instead.\nSentence 6: The briefing with Ines was moved to Monday at 9:00 in the south conference room.\nSentence 7: Tariq suggested meeting on Friday at 11:00 in the south room instead.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_expert_024",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The secondary sensor near central indicated roughly 45.2 mg/L.\nSentence 2: An uncalibrated device at riverside showed approximately 14.6 kPa.\nSentence 3: The backup instrument at riverside registered 18.9 lux before recalibration.\nSentence 4: According to the verified sensor at riverside, the measurement was 67.4 kPa.\nSentence 5: An uncalibrated device at downtown showed approximately 31.8 lux.\nSentence 6: A portable meter at riverside gave a reading of 31.8 kPa.\nSentence 7: A portable meter at downtown gave a reading of 31.8 kPa.\nSentence 8: The temporary sensor installed at downtown read 67.4 mg/L.\nSentence 9: The secondary sensor near coastal indicated roughly 45.2 kPa.\n\nAccording to sentence 4 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"67.4 kPa\"}"
 },
 {
  "task_id": "flanker_expert_025",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Shipment #F-9954 with 420 units of panels was redirected to Gdansk.\nSentence 2: The preliminary order #F-9954 allocated 350 units of panels to Valetta.\nSentence 3: Order #B-2293 for 120 units of components has been confirmed for delivery to Oulu.\nSentence 4: Draft manifest #A-4017 tentatively lists 85 units of filters for Luang Prabang.\nSentence 5: The preliminary order #F-9954 allocated 350 units of panels to Mandalay.\nSentence 6: Shipment #F-9954 with 350 units of brackets was redirected to Mandalay.\nSentence 7: Cancelled order #C-8841 had specified 420 units of modules for Tbilisi.\nSentence 8: The preliminary order #A-4017 allocated 230 units of adapters to Cartagena.\nSentence 9: Cancelled order #E-7120 had specified 230 units of adapters for Ulaanbaatar.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_expert_026",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Celine suggested meeting on Thursday at 14:30 in the north room instead.\nSentence 2: The deadline proposed by Celine is Friday, with the review starting at 13:00.\nSentence 3: The deadline set by Joaquin falls on Saturday, and the review begins at 10:30.\nSentence 4: Lumi requested a reschedule to Saturday at 13:00 in room 305.\nSentence 5: The briefing with Zora was moved to Friday at 14:30 in the main conference room.\nSentence 6: Tala requested a reschedule to Saturday at 16:00 in room 410.\nSentence 7: The follow-up with Hana was postponed to Friday at 9:00 in room 603.\nSentence 8: The deadline proposed by Ravi is Monday, with the review starting at 14:30.\nSentence 9: Lumi requested a reschedule to Friday at 13:00 in room 507.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_expert_027",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #A-4017 tentatively lists 120 units of filters for Tbilisi.\nSentence 2: Cancelled order #D-3506 had specified 120 units of components for Ulaanbaatar.\nSentence 3: Shipment #D-3506 with 510 units of adapters was redirected to Luang Prabang.\nSentence 4: Draft manifest #E-7120 tentatively lists 420 units of modules for Plovdiv.\nSentence 5: A revised order #C-8841 for 65 units of components is pending approval for Gdansk.\nSentence 6: Shipment #D-3506 containing 350 units of panels was dispatched to Tbilisi.\nSentence 7: Cancelled order #E-7120 had specified 120 units of brackets for Luang Prabang.\nSentence 8: Manifest #B-2293 was amended to show 350 units of panels for Recife.\nSentence 9: Shipment #B-2293 with 120 units of brackets was redirected to Cusco.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_expert_028",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Manifest #F-9954 was amended to show 175 units of filters for Reykjavik.\nSentence 2: The preliminary order #E-7120 allocated 120 units of modules to Ulaanbaatar.\nSentence 3: Draft manifest #D-3506 tentatively lists 85 units of components for Jaipur.\nSentence 4: Manifest #D-3506 was amended to show 175 units of panels for Valetta.\nSentence 5: Draft manifest #A-4017 tentatively lists 175 units of components for Gdansk.\nSentence 6: Shipment #A-4017 containing 510 units of brackets was dispatched to Trieste.\nSentence 7: The preliminary order #E-7120 allocated 85 units of components to Luang Prabang.\nSentence 8: The preliminary order #E-7120 allocated 65 units of components to Cartagena.\nSentence 9: Cancelled order #E-7120 had specified 350 units of adapters for Tallinn.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_expert_029",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #B-2293 tentatively lists 510 units of filters for Cusco.\nSentence 2: Manifest #B-2293 was amended to show 85 units of modules for Jaipur.\nSentence 3: Draft manifest #A-4017 tentatively lists 230 units of adapters for Mandalay.\nSentence 4: A revised order #D-3506 for 65 units of components is pending approval for Reykjavik.\nSentence 5: Shipment #B-2293 containing 420 units of filters was dispatched to Mandalay.\nSentence 6: A revised order #A-4017 for 85 units of brackets is pending approval for Ulaanbaatar.\nSentence 7: The preliminary order #B-2293 allocated 120 units of adapters to Plovdiv.\nSentence 8: Draft manifest #C-8841 tentatively lists 420 units of adapters for Luang Prabang.\nSentence 9: A revised order #C-8841 for 350 units of adapters is pending approval for Mandalay.\n\nAccording to sentence 5 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"420\"}"
 },
 {
  "task_id": "flanker_expert_030",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Haruto was moved to Monday at 9:00 in the north conference room.\nSentence 2: The briefing with Nico was moved to Wednesday at 11:00 in the main conference room.\nSentence 3: The briefing with Nalini was moved to Monday at 15:00 in the south conference room.\nSentence 4: The deadline set by Dmitri falls on Saturday, and the review begins at 16:00.\nSentence 5: Greta suggested meeting on Friday at 11:00 in the north room instead.\nSentence 6: A preliminary session with Dariush is tentatively set for Saturday at 10:30.\nSentence 7: A preliminary session with Nalini is tentatively set for Monday at 16:00.\nSentence 8: The deadline proposed by Zora is Thursday, with the review starting at 10:30.\nSentence 9: The follow-up with Qadir was postponed to Friday at 11:00 in room 603.\n\nAccording to sentence 4 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_expert_031",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #A-4017 tentatively lists 65 units of adapters for Oulu.\nSentence 2: Manifest #B-2293 was amended to show 350 units of components for Zanzibar.\nSentence 3: Order #A-4017 for 350 units of modules has been confirmed for delivery to Reykjavik.\nSentence 4: The preliminary order #A-4017 allocated 65 units of adapters to Luang Prabang.\nSentence 5: A revised order #B-2293 for 65 units of components is pending approval for Kumasi.\nSentence 6: Manifest #F-9954 was amended to show 65 units of components for Zanzibar.\nSentence 7: The preliminary order #A-4017 allocated 65 units of panels to Tbilisi.\nSentence 8: The preliminary order #E-7120 allocated 350 units of components to Tallinn.\nSentence 9: Shipment #D-3506 with 85 units of adapters was redirected to Tallinn.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_032",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Cancelled order #E-7120 had specified 420 units of modules for Luang Prabang.\n2. Shipment #F-9954 with 350 units of panels was redirected to Gdansk.\n3. Draft manifest #A-4017 tentatively lists 120 units of panels for Bruges.\n4. Shipment #D-3506 with 120 units of adapters was redirected to Recife.\n5. Shipment #B-2293 with 510 units of modules was redirected to Ulaanbaatar.\n6. Shipment #B-2293 with 350 units of adapters was redirected to Reykjavik.\n7. Draft manifest #D-3506 tentatively lists 120 units of components for Gdansk.\n8. Manifest #A-4017 lists 510 units of modules bound for Zanzibar.\n9. The preliminary order #F-9954 allocated 65 units of filters to Bruges.\n10. The preliminary order #E-7120 allocated 420 units of filters to Bruges.\n11. Manifest #F-9954 was amended to show 420 units of filters for Gdansk.\n12. Draft manifest #F-9954 tentatively lists 230 units of adapters for Tallinn.\n13. A revised order #B-2293 for 175 units of panels is pending approval for Ulaanbaatar.\n\nAccording to sentence 8 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_frontier_033",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. An older model gauge at central displayed 14.6 °C.\n2. An older model gauge at downtown displayed 45.2 kPa.\n3. The backup instrument at downtown registered 45.2 kPa before recalibration.\n4. The temporary sensor installed at coastal read 23.7 kPa.\n5. The calibrated instrument recorded a reading of 14.6 mV at the coastal station.\n6. A portable meter at northern gave a reading of 18.9 mV.\n7. An older model gauge at northern displayed 67.4 kPa.\n8. An older model gauge at central displayed 31.8 mg/L.\n9. The secondary sensor near northern indicated roughly 52.1 mV.\n10. The backup instrument at coastal registered 23.7 lux before recalibration.\n11. An older model gauge at downtown displayed 14.6 ppm.\n12. An older model gauge at central displayed 52.1 kPa.\n13. The temporary sensor installed at coastal read 52.1 mV.\n\nAccording to sentence 5 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"14.6 mV\"}"
 },
 {
  "task_id": "flanker_frontier_034",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Draft manifest #E-7120 tentatively lists 420 units of adapters for Cartagena.\n2. Manifest #B-2293 was amended to show 120 units of modules for Trieste.\n3. The preliminary order #E-7120 allocated 350 units of brackets to Gdansk.\n4. The preliminary order #B-2293 allocated 65 units of brackets to Tallinn.\n5. Manifest #B-2293 was amended to show 85 units of modules for Luang Prabang.\n6. Shipment #E-7120 containing 350 units of filters was dispatched to Luang Prabang.\n7. A revised order #F-9954 for 420 units of adapters is pending approval for Bruges.\n8. Manifest #F-9954 was amended to show 350 units of components for Luang Prabang.\n9. A revised order #B-2293 for 85 units of components is pending approval for Recife.\n10. Cancelled order #A-4017 had specified 510 units of brackets for Cartagena.\n11. Manifest #F-9954 was amended to show 230 units of filters for Kumasi.\n12. A revised order #C-8841 for 120 units of adapters is pending approval for Kotor.\n13. Cancelled order #E-7120 had specified 420 units of adapters for Trieste.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_035",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Cancelled order #C-8841 had specified 65 units of modules for Tbilisi.\n2. Cancelled order #D-3506 had specified 175 units of filters for Bruges.\n3. A revised order #F-9954 for 120 units of filters is pending approval for Mandalay.\n4. Cancelled order #D-3506 had specified 510 units of components for Jaipur.\n5. Shipment #B-2293 with 85 units of modules was redirected to Oulu.\n6. Manifest #B-2293 lists 350 units of modules bound for Reykjavik.\n7. Manifest #B-2293 was amended to show 230 units of filters for Trieste.\n8. Shipment #E-7120 with 85 units of panels was redirected to Plovdiv.\n9. A revised order #C-8841 for 120 units of panels is pending approval for Plovdiv.\n10. The preliminary order #A-4017 allocated 85 units of adapters to Kotor.\n11. Shipment #E-7120 with 510 units of brackets was redirected to Fez.\n12. Shipment #E-7120 with 420 units of filters was redirected to Fez.\n13. Cancelled order #E-7120 had specified 120 units of panels for Jaipur.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_036",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Manifest #E-7120 was amended to show 230 units of adapters for Trieste.\n2. Cancelled order #B-2293 had specified 420 units of components for Gdansk.\n3. Manifest #F-9954 was amended to show 65 units of components for Trieste.\n4. Manifest #F-9954 was amended to show 420 units of brackets for Zanzibar.\n5. Manifest #F-9954 was amended to show 65 units of brackets for Ulaanbaatar.\n6. Shipment #A-4017 with 350 units of components was redirected to Valetta.\n7. Shipment #B-2293 containing 85 units of panels was dispatched to Tallinn.\n8. A revised order #B-2293 for 175 units of panels is pending approval for Ulaanbaatar.\n9. Shipment #E-7120 with 230 units of components was redirected to Zanzibar.\n10. A revised order #E-7120 for 65 units of filters is pending approval for Ulaanbaatar.\n11. Draft manifest #B-2293 tentatively lists 350 units of components for Recife.\n12. Draft manifest #D-3506 tentatively lists 85 units of modules for Trieste.\n13. Cancelled order #F-9954 had specified 350 units of filters for Luang Prabang.\n\nAccording to sentence 7 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"85\"}"
 },
 {
  "task_id": "flanker_frontier_037",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. A portable meter at downtown gave a reading of 76.3 mV.\n2. An older model gauge at downtown displayed 31.8 kPa.\n3. An older model gauge at riverside displayed 23.7 mV.\n4. An older model gauge at northern displayed 67.4 °C.\n5. The temporary sensor installed at central read 76.3 ppm.\n6. The temporary sensor installed at northern read 23.7 kPa.\n7. The official reading from downtown showed 52.1 ppm on the certified gauge.\n8. The secondary sensor near riverside indicated roughly 18.9 °C.\n9. The secondary sensor near hilltop indicated roughly 31.8 kPa.\n10. An older model gauge at central displayed 18.9 kPa.\n11. The temporary sensor installed at central read 23.7 mg/L.\n12. The backup instrument at downtown registered 18.9 lux before recalibration.\n13. An older model gauge at central displayed 18.9 mg/L.\n\nAccording to sentence 7 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"52.1 ppm\"}"
 },
 {
  "task_id": "flanker_frontier_038",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Draft manifest #A-4017 tentatively lists 175 units of adapters for Trieste.\n2. Cancelled order #C-8841 had specified 120 units of filters for Cusco.\n3. Cancelled order #B-2293 had specified 350 units of panels for Tallinn.\n4. The preliminary order #C-8841 allocated 350 units of filters to Gdansk.\n5. The preliminary order #C-8841 allocated 230 units of filters to Recife.\n6. A revised order #F-9954 for 420 units of adapters is pending approval for Mandalay.\n7. The preliminary order #C-8841 allocated 120 units of filters to Trieste.\n8. Manifest #B-2293 lists 350 units of modules bound for Fez.\n9. Manifest #B-2293 was amended to show 65 units of modules for Bruges.\n10. Shipment #D-3506 with 350 units of adapters was redirected to Zanzibar.\n11. Cancelled order #E-7120 had specified 175 units of brackets for Kotor.\n12. The preliminary order #A-4017 allocated 120 units of adapters to Recife.\n13. Manifest #D-3506 was amended to show 65 units of components for Bruges.\n\nAccording to sentence 8 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_039",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. The backup instrument at downtown registered 52.1 lux before recalibration.\n2. The secondary sensor near riverside indicated roughly 31.8 °C.\n3. The secondary sensor near central indicated roughly 23.7 kPa.\n4. An uncalibrated device at hilltop showed approximately 14.6 ppm.\n5. According to the verified sensor at central, the measurement was 31.8 kPa.\n6. The backup instrument at coastal registered 76.3 mV before recalibration.\n7. An older model gauge at riverside displayed 23.7 mg/L.\n8. A portable meter at riverside gave a reading of 18.9 mg/L.\n9. An uncalibrated device at downtown showed approximately 67.4 lux.\n10. The backup instrument at downtown registered 45.2 mg/L before recalibration.\n11. The temporary sensor installed at central read 31.8 mV.\n12. An older model gauge at coastal displayed 52.1 mV.\n13. The backup instrument at riverside registered 52.1 kPa before recalibration.\n\nAccording to sentence 5 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"31.8 kPa\"}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['selective', 'stroop', 'flanker']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "selective": cogattention_selective,
    "stroop": cogattention_stroop,
    "flanker": cogattention_flanker,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Selective Attention")
